In [ ]:
# ============================================================
# CELL 1 — TASK:
# Prepare a fresh Kaggle environment for the full reviewer
# revision experiment covering Steps 1–4.
# ============================================================

!pip install -q --upgrade-strategy only-if-needed \
    "transformers>=5.5.0" \
    "accelerate>=1.10.0" \
    "bitsandbytes>=0.48.0" \
    "sentencepiece>=0.2.0"

import os
import re
import gc
import json
import time
import glob
import math
import random
import zipfile
import warnings
import textwrap
import unicodedata

from pathlib import Path
from itertools import combinations

os.environ.setdefault(
    "PYTORCH_CUDA_ALLOC_CONF",
    "expandable_segments:True"
)
os.environ.setdefault(
    "TOKENIZERS_PARALLELISM",
    "false"
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from scipy.stats import binomtest
from sklearn.metrics import (
    f1_score,
    cohen_kappa_score,
)

warnings.filterwarnings("ignore")

PRIMARY_SEED = 42
N_BOOT = 5000
N_PERM = 10000

random.seed(PRIMARY_SEED)
np.random.seed(PRIMARY_SEED)

if torch.cuda.is_available():
    torch.manual_seed(PRIMARY_SEED)
    torch.cuda.manual_seed_all(PRIMARY_SEED)
    torch.backends.cuda.matmul.allow_tf32 = True


# ------------------------------------------------------------
# Detect damaged Kaggle PyTorch stack
# ------------------------------------------------------------

def check_kaggle_torch_stack():

    problems = []

    try:
        import torchvision
        _ = torchvision.__version__
    except Exception as e:
        problems.append(
            f"torchvision import failed: {e}"
        )

    try:
        import torchaudio
        _ = torchaudio.__version__
    except Exception as e:
        problems.append(
            f"torchaudio import failed: {e}"
        )

    if problems:

        raise RuntimeError(
            "Kaggle PyTorch/CUDA environment is inconsistent.\n"
            "Restart with a FRESH Kaggle session.\n\n"
            + "\n".join(problems)
        )


check_kaggle_torch_stack()


from transformers import (
    AutoTokenizer,
    AutoProcessor,
    AutoModelForCausalLM,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
)


OUT_DIR = Path(
    "/kaggle/working/ambitab_full_revision_v7"
)
OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CKPT_DIR = OUT_DIR / "checkpoints"
CKPT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


LOAD_IN_4BIT = (
    os.environ.get(
        "LOAD_IN_4BIT",
        "0"
    ) == "1"
)

MAX_INPUT_TOKENS = int(
    os.environ.get(
        "MAX_INPUT_TOKENS",
        "2048"
    )
)


REAL_MODELS = [
    "Qwen2.5-7B-Instruct",
    "Gemma-4-E4B-it",
    "Mistral-7B-Instruct-v0.2",
]

print("✓ Environment ready")
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}:",
        torch.cuda.get_device_name(i)
    )

In [ ]:
# ============================================================
# CELL 2 — TASK:
# Locate and validate the REAL benchmark and human-evaluation
# files inside Kaggle inputs, including a nested repository.
#
# IMPORTANT:
# This cell rejects placeholder benchmark CSVs that do not
# contain the full released benchmark schema.
# ============================================================

from pathlib import Path
import pandas as pd
import glob

REQUIRED_BENCHMARK_COLUMNS = {
    "query_id",
    "query_text",
    "final_query_type",
    "subdomain",
    "gold_answer_1",
}

REQUIRED_HUMAN_COLUMNS = {
    "query_id",
    "model",
    "subdomain",
    "final_interpretation_validity",
    "final_assumption_quality",
    "final_answer_usefulness",
    "human_total",
    "human_score",
}


def _csv_header(path):
    """Read only the header of a CSV safely."""
    try:
        tmp = pd.read_csv(path, nrows=0, encoding="utf-8-sig")
        return {str(c).strip() for c in tmp.columns}
    except Exception:
        return set()


def find_valid_csv(patterns, required_columns, expected_rows=None, label="file"):
    """
    Search recursively under /kaggle/input and return the first
    CSV that satisfies the required schema (and row count when
    supplied). Placeholder files are skipped automatically.
    """
    candidates = []

    for pattern in patterns:
        candidates.extend(
            glob.glob(
                f"/kaggle/input/**/{pattern}",
                recursive=True,
            )
        )

    # de-duplicate while preserving deterministic order
    candidates = sorted(dict.fromkeys(candidates))

    rejected = []

    for path in candidates:
        cols = _csv_header(path)

        if not required_columns.issubset(cols):
            rejected.append((path, sorted(cols)))
            continue

        if expected_rows is not None:
            try:
                n_rows = len(pd.read_csv(path, usecols=["query_id"], encoding="utf-8-sig"))
            except Exception:
                rejected.append((path, sorted(cols)))
                continue

            if n_rows != expected_rows:
                rejected.append((path, sorted(cols)))
                continue

        return path

    print(f"\nNo valid {label} was found.")

    if rejected:
        print("\nFiles found but rejected:")
        for path, cols in rejected:
            print(" -", path)
            print("   columns:", cols)

    raise FileNotFoundError(
        f"A valid {label} is required.\n"
        f"Required columns: {sorted(required_columns)}\n"
        + (f"Expected rows: {expected_rows}\n" if expected_rows is not None else "")
        + "\nIf you uploaded the repository template, replace its placeholder benchmark CSV with the real full benchmark file, or attach the real benchmark as a separate Kaggle dataset."
    )


def find_optional_file(patterns):
    candidates = []
    for pattern in patterns:
        candidates.extend(
            glob.glob(
                f"/kaggle/input/**/{pattern}",
                recursive=True,
            )
        )
    candidates = sorted(dict.fromkeys(candidates))
    return candidates[0] if candidates else None


# ------------------------------------------------------------
# REAL 360-row benchmark
# ------------------------------------------------------------

BENCH_PATH = find_valid_csv(
    patterns=[
        "AmbiTab_Sports_Full_Gold_Benchmark_FINAL.csv",
        "AmbiTab_Sports_Full_Gold_Benchmark_v1.csv",
        "AmbiTab_Sports_Full_Gold_Benchmark_v3.csv",
        "*AmbiTab*Full*Gold*Benchmark*.csv",
    ],
    required_columns=REQUIRED_BENCHMARK_COLUMNS,
    expected_rows=360,
    label="360-row full benchmark",
)


# ------------------------------------------------------------
# Final 480-row cooperative human gold
# ------------------------------------------------------------

HUMAN_PATH = find_valid_csv(
    patterns=[
        "cooperative_human_gold_final_480.csv",
        "*cooperative*human*gold*final*480*.csv",
    ],
    required_columns=REQUIRED_HUMAN_COLUMNS,
    expected_rows=480,
    label="480-row cooperative human-gold file",
)


# ------------------------------------------------------------
# Optional cooperative annotator files
# ------------------------------------------------------------

ANNOTATOR_A_PATH = find_optional_file([
    "cooperative_annotator_A_FILLED.csv",
    "cooperative_annotator_A_FILLED(1).csv",
    "cooperative_annotator_A_FILLED(2).csv",
    "*cooperative*annotator*A*FILLED*.csv",
])

ANNOTATOR_B_PATH = find_optional_file([
    "cooperative_annotator_B_FILLED.csv",
    "cooperative_annotator_B_FILLED(1).csv",
    "*cooperative*annotator*B*FILLED*.csv",
])


# ------------------------------------------------------------
# Optional authentic table evidence only
# ------------------------------------------------------------

TABLE_EVIDENCE_PATH = find_optional_file([
    "AmbiTab_Sports_Table_Evidence.csv",
    "*AmbiTab*Table*Evidence*.csv",
])


# ------------------------------------------------------------
# Load and validate benchmark
# ------------------------------------------------------------

df = pd.read_csv(
    BENCH_PATH,
    encoding="utf-8-sig",
)

df.columns = [
    str(c).strip()
    for c in df.columns
]

missing = REQUIRED_BENCHMARK_COLUMNS - set(df.columns)
if missing:
    raise KeyError(
        f"Benchmark missing required columns: {sorted(missing)}"
    )

assert len(df) == 360, f"Expected 360 benchmark rows, found {len(df)}"
assert df["query_id"].nunique() == 360, "query_id values must be unique"

track_U = (
    df[
        df["final_query_type"].astype(str).str.strip().eq("unambiguous")
    ]
    .copy()
    .reset_index(drop=True)
)

track_C = (
    df[
        df["final_query_type"].astype(str).str.strip().eq("cooperative_ambiguous")
    ]
    .copy()
    .reset_index(drop=True)
)

track_X = (
    df[
        df["final_query_type"].astype(str).str.strip().eq("uncooperative")
    ]
    .copy()
    .reset_index(drop=True)
)

assert (len(track_U), len(track_C), len(track_X)) == (120, 120, 120), (
    "Expected 120 items in each track, found "
    f"U={len(track_U)}, C={len(track_C)}, X={len(track_X)}"
)


# ------------------------------------------------------------
# Load and validate final cooperative human evaluation
# ------------------------------------------------------------

human_gold = pd.read_csv(
    HUMAN_PATH,
    encoding="utf-8-sig",
)

human_gold.columns = [
    str(c).strip()
    for c in human_gold.columns
]

missing_h = REQUIRED_HUMAN_COLUMNS - set(human_gold.columns)
if missing_h:
    raise KeyError(
        "Human gold missing required columns: "
        + str(sorted(missing_h))
    )

assert len(human_gold) == 480, f"Expected 480 human responses, found {len(human_gold)}"
assert human_gold["query_id"].nunique() == 120, (
    "Expected 120 unique cooperative queries in the human-gold file"
)


print("✓ VALID benchmark:", BENCH_PATH)
print("✓ VALID cooperative human gold:", HUMAN_PATH)
print("✓ Benchmark rows:", len(df))
print("✓ Tracks: U=120, C=120, X=120")
print("✓ Human cooperative responses:", len(human_gold))
print("Annotator A:", ANNOTATOR_A_PATH)
print("Annotator B:", ANNOTATOR_B_PATH)
print("Authentic table evidence:", TABLE_EVIDENCE_PATH)


In [ ]:
# ============================================================
# CELL 3 — TASK:
# Configure and load the exact model checkpoints used in the
# revised experiment. Models are loaded one at a time so the
# notebook can run safely on Kaggle T4 x2.
# ============================================================

try:

    from kaggle_secrets import UserSecretsClient

    _secrets = UserSecretsClient()

    def get_secret(
        name,
        default=""
    ):

        try:
            return _secrets.get_secret(name)

        except Exception:
            return os.environ.get(
                name,
                default
            )

except Exception:

    def get_secret(
        name,
        default=""
    ):
        return os.environ.get(
            name,
            default
        )


HF_TOKEN = (
    get_secret("HF_TOKEN")
    or
    get_secret("HUGGINGFACE_TOKEN")
    or
    os.environ.get("HF_TOKEN", "")
)


if HF_TOKEN:

    try:

        from huggingface_hub import login

        login(
            token=HF_TOKEN,
            add_to_git_credential=False,
        )

        print("✓ Hugging Face login ready")

    except Exception as e:

        print(
            "HF login warning:",
            e
        )

else:

    print(
        "! No HF_TOKEN found."
    )


MODEL_CONFIGS = {

    "Qwen2.5-7B-Instruct": {

        "model_id":
            "Qwen/Qwen2.5-7B-Instruct",

        "preprocessor_kind":
            "tokenizer",

        "model_family":
            "causal_lm",
    },

    "Gemma-4-E4B-it": {

        "model_id":
            "google/gemma-4-E4B-it",

        "preprocessor_kind":
            "processor",

        "model_family":
            "multimodal_lm",
    },

    "Mistral-7B-Instruct-v0.2": {

        "model_id":
            "mistralai/Mistral-7B-Instruct-v0.2",

        "preprocessor_kind":
            "tokenizer",

        "model_family":
            "causal_lm",
    },
}


manifest = pd.DataFrame([
    {
        "display_name": name,
        **config,
    }
    for name, config
    in MODEL_CONFIGS.items()
])


manifest.to_csv(
    OUT_DIR /
    "model_manifest_exact_checkpoints.csv",
    index=False,
)

display(manifest)


CACHE = {
    "display_name": None,
    "preprocessor": None,
    "model": None,
    "kind": None,
}


def retry(
    fn,
    attempts=3,
    base_wait=2.0,
):

    last_error = None

    for attempt in range(
        attempts
    ):

        try:
            return fn()

        except Exception as e:

            last_error = e

            if attempt == attempts - 1:
                raise

            wait = (
                base_wait
                * (2 ** attempt)
                + random.random()
            )

            print(
                f"Retry after "
                f"{type(e).__name__}: {e}"
            )

            time.sleep(wait)

    raise last_error


def clear_local_model():

    model = CACHE.get(
        "model"
    )

    prep = CACHE.get(
        "preprocessor"
    )

    CACHE.update({
        "display_name": None,
        "preprocessor": None,
        "model": None,
        "kind": None,
    })

    del model
    del prep

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def preferred_dtype():

    if not torch.cuda.is_available():
        return torch.float32

    if torch.cuda.is_bf16_supported():
        return torch.bfloat16

    return torch.float16


def processor_tokenizer(prep):

    return getattr(
        prep,
        "tokenizer",
        prep
    )


def load_local_model(
    display_name
):

    if (
        CACHE["display_name"]
        == display_name
        and
        CACHE["model"]
        is not None
    ):

        return (
            CACHE["preprocessor"],
            CACHE["model"],
            CACHE["kind"],
        )


    clear_local_model()

    cfg = MODEL_CONFIGS[
        display_name
    ]

    model_id = cfg[
        "model_id"
    ]

    kind = cfg[
        "preprocessor_kind"
    ]

    family = cfg[
        "model_family"
    ]

    dtype = preferred_dtype()

    print(
        f"\nLoading {display_name}"
    )

    print(
        "Checkpoint:",
        model_id
    )


    if kind == "processor":

        prep = AutoProcessor.from_pretrained(
            model_id,
            token=HF_TOKEN or None,
            trust_remote_code=True,
        )

    else:

        prep = AutoTokenizer.from_pretrained(
            model_id,
            token=HF_TOKEN or None,
            use_fast=True,
            trust_remote_code=True,
        )

        if (
            getattr(
                prep,
                "pad_token",
                None
            )
            is None
        ):

            prep.pad_token = (
                prep.eos_token
            )


    kwargs = {

        "token":
            HF_TOKEN or None,

        "low_cpu_mem_usage":
            True,

        "trust_remote_code":
            True,
    }


    if torch.cuda.is_available():

        kwargs[
            "device_map"
        ] = "auto"


    if (
        LOAD_IN_4BIT
        and
        torch.cuda.is_available()
    ):

        kwargs[
            "quantization_config"
        ] = BitsAndBytesConfig(

            load_in_4bit=True,

            bnb_4bit_compute_dtype=
                dtype,

            bnb_4bit_quant_type=
                "nf4",

            bnb_4bit_use_double_quant=
                True,
        )

    else:

        kwargs[
            "dtype"
        ] = (
            dtype
            if torch.cuda.is_available()
            else torch.float32
        )


    model_class = (
        AutoModelForMultimodalLM
        if family
        == "multimodal_lm"
        else
        AutoModelForCausalLM
    )


    try:

        model = (
            model_class
            .from_pretrained(
                model_id,
                **kwargs
            )
        )

    except TypeError as e:

        if "dtype" in str(e):

            kwargs[
                "torch_dtype"
            ] = kwargs.pop(
                "dtype"
            )

            model = (
                model_class
                .from_pretrained(
                    model_id,
                    **kwargs
                )
            )

        else:
            raise


    model.eval()


    CACHE.update({

        "display_name":
            display_name,

        "preprocessor":
            prep,

        "model":
            model,

        "kind":
            kind,
    })


    return (
        prep,
        model,
        kind,
    )


def chat_prompt(
    prep,
    system,
    user,
    kind,
):

    if kind == "processor":

        combined = (
            f"{system}\n\n{user}"
        )

        messages = [{
            "role": "user",
            "content": [{
                "type": "text",
                "text": combined,
            }],
        }]

        try:

            return prep.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

        except Exception:

            return (
                f"User: {combined}\n\n"
                f"Assistant:"
            )


    messages = [
        {
            "role": "system",
            "content": system,
        },
        {
            "role": "user",
            "content": user,
        },
    ]


    try:

        return prep.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    except Exception:

        combined = (
            f"{system}\n\n{user}"
        )

        try:

            return prep.apply_chat_template(
                [{
                    "role": "user",
                    "content": combined,
                }],
                tokenize=False,
                add_generation_prompt=True,
            )

        except Exception:

            return (
                f"System: {system}\n\n"
                f"User: {user}\n\n"
                f"Assistant:"
            )


def prepare_inputs(
    prep,
    prompt,
    kind,
):

    if kind == "processor":

        try:

            return prep(
                text=prompt,
                return_tensors="pt",
                truncation=True,
                max_length=
                    MAX_INPUT_TOKENS,
            )

        except TypeError:

            return prep(
                text=prompt,
                return_tensors="pt",
            )


    return prep(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=
            MAX_INPUT_TOKENS,
    )


def get_token_id(
    prep,
    name,
):

    value = getattr(
        prep,
        name,
        None
    )

    if value is None:

        value = getattr(
            processor_tokenizer(prep),
            name,
            None,
        )

    return value


def call_local_text(
    display_name,
    system,
    user,
    max_output_tokens=128,
):

    def run():

        prep, model, kind = (
            load_local_model(
                display_name
            )
        )

        prompt = chat_prompt(
            prep,
            system,
            user,
            kind,
        )

        inputs = prepare_inputs(
            prep,
            prompt,
            kind,
        )


        try:

            device = (
                model
                .get_input_embeddings()
                .weight
                .device
            )

        except Exception:

            device = next(
                model.parameters()
            ).device


        inputs = {
            k: (
                v.to(device)
                if hasattr(v, "to")
                else v
            )
            for k, v
            in inputs.items()
        }


        generation_kwargs = {

            "max_new_tokens":
                max_output_tokens,

            "do_sample":
                False,
        }


        pad_id = get_token_id(
            prep,
            "pad_token_id"
        )

        eos_id = get_token_id(
            prep,
            "eos_token_id"
        )


        if pad_id is not None:
            generation_kwargs[
                "pad_token_id"
            ] = pad_id

        if eos_id is not None:
            generation_kwargs[
                "eos_token_id"
            ] = eos_id


        with torch.inference_mode():

            output = model.generate(
                **inputs,
                **generation_kwargs,
            )


        input_length = (
            inputs[
                "input_ids"
            ]
            .shape[-1]
        )


        generated = (
            output[0][
                input_length:
            ]
        )


        return (
            processor_tokenizer(
                prep
            )
            .decode(
                generated,
                skip_special_tokens=True,
            )
            .strip()
        )


    try:

        return retry(run)

    except Exception as e:

        return (
            f"ERROR: "
            f"{type(e).__name__}: "
            f"{e}"
        )


print("✓ Model runner ready")

In [ ]:
# ============================================================
# CELL 4 — TASK:
# Define the statistical tests used for Steps 1–3:
# bootstrap confidence intervals, paired differences,
# permutation tests, McNemar tests, and Holm correction.
# ============================================================

def bootstrap_mean_ci(
    values,
    n_boot=N_BOOT,
    seed=PRIMARY_SEED,
):

    x = np.asarray(
        values,
        dtype=float
    )

    rng = np.random.default_rng(
        seed
    )

    indices = rng.integers(
        0,
        len(x),
        size=(
            n_boot,
            len(x)
        ),
    )

    means = (
        x[indices]
        .mean(axis=1)
    )

    return (
        float(x.mean()),
        float(
            np.quantile(
                means,
                0.025
            )
        ),
        float(
            np.quantile(
                means,
                0.975
            )
        ),
    )


def bootstrap_diff(
    a,
    b,
    n_boot=N_BOOT,
    seed=PRIMARY_SEED,
):

    d = (
        np.asarray(
            a,
            dtype=float
        )
        -
        np.asarray(
            b,
            dtype=float
        )
    )

    rng = np.random.default_rng(
        seed
    )

    indices = rng.integers(
        0,
        len(d),
        size=(
            n_boot,
            len(d)
        ),
    )

    vals = (
        d[indices]
        .mean(axis=1)
    )

    return (
        float(d.mean()),
        float(
            np.quantile(
                vals,
                0.025
            )
        ),
        float(
            np.quantile(
                vals,
                0.975
            )
        ),
    )


def paired_cohen_dz(
    a,
    b,
):

    d = (
        np.asarray(
            a,
            dtype=float
        )
        -
        np.asarray(
            b,
            dtype=float
        )
    )

    sd = d.std(
        ddof=1
    )

    if sd == 0:

        if abs(d.mean()) < 1e-15:
            return 0.0

        return np.inf

    return float(
        d.mean() / sd
    )


def holm_adjust(
    pvalues
):

    pvalues = np.asarray(
        pvalues,
        dtype=float
    )

    m = len(pvalues)

    order = np.argsort(
        pvalues
    )

    adjusted = np.zeros(
        m
    )

    running_max = 0.0

    for rank, index in enumerate(
        order
    ):

        candidate = (
            (m - rank)
            *
            pvalues[index]
        )

        running_max = max(
            running_max,
            candidate
        )

        adjusted[index] = min(
            1.0,
            running_max
        )

    return adjusted


def paired_permutation_p(
    a,
    b,
    n_perm=N_PERM,
    seed=PRIMARY_SEED,
):

    d = (
        np.asarray(
            a,
            dtype=float
        )
        -
        np.asarray(
            b,
            dtype=float
        )
    )

    observed = abs(
        d.mean()
    )

    rng = np.random.default_rng(
        seed
    )

    exceed = 0

    for _ in range(
        n_perm
    ):

        signs = rng.choice(
            [-1.0, 1.0],
            size=len(d),
        )

        statistic = abs(
            (
                d * signs
            ).mean()
        )

        if (
            statistic
            >= observed - 1e-15
        ):
            exceed += 1

    return (
        exceed + 1
    ) / (
        n_perm + 1
    )


print("✓ Statistical helpers ready")

In [ ]:
# ============================================================
# CELL 5 — TASK:
# STEP 1.
# Run fresh inference on ALL 120 unambiguous queries for the
# three LLMs. Checkpoint every 10 queries.
# ============================================================

from tqdm.auto import tqdm


EXEC_SYSTEM = """
You are a sports statistics assistant.

Answer the user's question directly and briefly.

Do not explain your reasoning.
Do not invent an answer if you are uncertain.
""".strip()


def execution_prompt(
    query
):

    return textwrap.dedent(
        f"""
        Query: {query}

        Return only the requested final answer.

        Rules:
        - For a person, team, club, country, player, or driver,
          return the requested name.
        - For a numerical question, return the requested number.
        - For a comparison, include all requested values.
        - For a list question, include every requested item.
        - Do not explain your reasoning.
        """
    ).strip()


EXEC_CKPT = (
    CKPT_DIR /
    "step1_execution_fresh_v7.json"
)


if EXEC_CKPT.exists():

    with open(
        EXEC_CKPT,
        "r"
    ) as f:
        exec_responses = json.load(f)

else:

    exec_responses = {
        model: [
            ""
        ] * len(track_U)
        for model
        in REAL_MODELS
    }


for model in REAL_MODELS:

    values = exec_responses.get(
        model,
        []
    )

    if len(values) < len(track_U):

        values += [
            ""
        ] * (
            len(track_U)
            - len(values)
        )

    exec_responses[
        model
    ] = values[
        :len(track_U)
    ]


for model in REAL_MODELS:

    pending = [
        i
        for i, value
        in enumerate(
            exec_responses[
                model
            ]
        )
        if not str(
            value
        ).strip()
    ]


    if not pending:

        print(
            f"✓ STEP 1 {model}: "
            "already complete"
        )

        continue


    load_local_model(
        model
    )


    for count, i in enumerate(

        tqdm(
            pending,
            desc=f"STEP 1 — {model}",
            unit="query",
        ),

        start=1,
    ):

        query = track_U.loc[
            i,
            "query_text"
        ]


        exec_responses[
            model
        ][i] = call_local_text(

            model,

            EXEC_SYSTEM,

            execution_prompt(
                query
            ),

            max_output_tokens=100,
        )


        if (
            count % 10 == 0
            or
            count == len(pending)
        ):

            with open(
                EXEC_CKPT,
                "w"
            ) as f:

                json.dump(
                    exec_responses,
                    f,
                )


    clear_local_model()


with open(
    EXEC_CKPT,
    "w"
) as f:

    json.dump(
        exec_responses,
        f,
    )


raw_rows = []


for model in REAL_MODELS:

    for i, row in (
        track_U.iterrows()
    ):

        raw_rows.append({

            "model":
                model,

            "query_id":
                row["query_id"],

            "subdomain":
                row["subdomain"],

            "query_text":
                row["query_text"],

            "raw_response":
                exec_responses[
                    model
                ][i],
        })


# Lower-bound descriptive baseline
for _, row in (
    track_U.iterrows()
):

    raw_rows.append({

        "model":
            "Heuristic-Baseline",

        "query_id":
            row["query_id"],

        "subdomain":
            row["subdomain"],

        "query_text":
            row["query_text"],

        "raw_response":
            "Unknown",
    })


raw_U = pd.DataFrame(
    raw_rows
)


raw_U.to_csv(
    OUT_DIR /
    "step1_raw_outputs_execution_v7.csv",
    index=False,
)


print(
    "✓ STEP 1 inference complete"
)

print(
    raw_U.groupby(
        "model"
    ).size()
)

In [ ]:
# ============================================================
# CELL 6 — TASK:
# STEP 1 scoring.
# Correct the old numeric-parser bug and prevent supporting
# statistics from being treated as alternative answers.
#
# Numeric questions -> primary number must match.
# Entity questions -> requested entity must match.
# Comparison questions -> both requested values must match.
# List questions -> all requested entities must appear.
# ============================================================

ORG_TOKENS = {
    "united",
    "states",
    "great",
    "britain",
    "manchester",
    "city",
    "real",
    "madrid",
    "bayern",
    "munich",
    "golden",
    "state",
    "warriors",
    "boston",
    "celtics",
    "milwaukee",
    "bucks",
    "red",
    "bull",
    "racing",
    "mclaren",
    "williams",
    "barcelona",
    "spain",
    "argentina",
    "england",
    "france",
    "croatia",
    "italy",
    "japan",
    "china",
    "mercedes",
}


def strip_accents(
    text
):

    return "".join(
        c
        for c
        in unicodedata.normalize(
            "NFKD",
            str(text)
        )
        if not unicodedata.combining(
            c
        )
    )


def normalize_text(
    text
):

    text = (
        strip_accents(
            text
        )
        .lower()
        .replace(
            "’",
            "'"
        )
    )

    text = re.sub(
        r"[^a-z0-9%+\-.\s]",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip()


def extract_numbers(
    text
):

    output = []

    pattern = (
        r"(?<![A-Za-z0-9])"
        r"([+-]?"
        r"(?:\d{1,3}(?:,\d{3})+|\d+)"
        r"(?:\.\d+)?)"
        r"(%)?"
    )

    for match in re.finditer(
        pattern,
        str(text)
    ):

        try:

            output.append(
                float(
                    match.group(1)
                    .replace(
                        ",",
                        ""
                    )
                )
            )

        except Exception:
            pass

    return output


def gold_numeric_value(
    gold
):

    match = re.match(

        r"^\s*"
        r"([+-]?"
        r"(?:\d{1,3}(?:,\d{3})+|\d+)"
        r"(?:\.\d+)?)",

        str(gold),
    )

    if not match:
        return None

    return float(
        match.group(1)
        .replace(
            ",",
            ""
        )
    )


def numeric_answer_present(
    response,
    gold,
    tolerance=0.02,
):

    gold_value = (
        gold_numeric_value(
            gold
        )
    )

    if gold_value is None:
        return False


    for predicted in extract_numbers(
        response
    ):

        # Exact
        if (
            abs(
                predicted
                -
                gold_value
            )
            <= 1e-9
        ):
            return True


        # Relative tolerance
        relative_error = (

            abs(
                predicted
                -
                gold_value
            )
            /
            max(
                abs(
                    gold_value
                ),
                1.0
            )
        )


        if (
            relative_error
            <= tolerance
        ):
            return True


    return False


def full_entity_present(
    response,
    gold
):

    response_norm = (
        normalize_text(
            response
        )
    )

    gold_norm = (
        normalize_text(
            gold
        )
    )

    if not gold_norm:
        return False


    pattern = (
        r"(?<![a-z0-9])"
        +
        re.escape(
            gold_norm
        )
        +
        r"(?![a-z0-9])"
    )


    return (
        re.search(
            pattern,
            response_norm
        )
        is not None
    )


def surname_with_support(
    response,
    gold_entity,
    supporting_gold,
):

    if (
        pd.isna(
            supporting_gold
        )
    ):
        return False


    words = (
        normalize_text(
            gold_entity
        )
        .split()
    )


    if len(words) < 2:
        return False


    surname = words[-1]


    if (
        len(surname) < 4
        or
        surname in ORG_TOKENS
    ):
        return False


    surname_found = (
        re.search(
            r"\b"
            +
            re.escape(
                surname
            )
            +
            r"\b",
            normalize_text(
                response
            ),
        )
        is not None
    )


    return (
        surname_found
        and
        numeric_answer_present(
            response,
            supporting_gold
        )
    )


def clean_list_entity(
    item
):

    return re.sub(
        r"\s*\([^)]*\)\s*$",
        "",
        str(item),
    ).strip()


def classify_execution_task(
    query,
    gold_1
):

    q = str(
        query
    ).strip().lower()


    if q.startswith(
        "list "
    ):
        return "list"


    if q.startswith(
        "compare "
    ):
        return "compare"


    if (
        gold_numeric_value(
            gold_1
        )
        is not None
    ):
        return "numeric"


    return "entity"


def score_execution_row(
    row
):

    response = str(
        row[
            "raw_response"
        ]
    )

    query = str(
        row[
            "query_text"
        ]
    )

    gold_1 = row[
        "gold_answer_1"
    ]

    gold_2 = row.get(
        "gold_answer_2",
        np.nan
    )


    task_type = (
        classify_execution_task(
            query,
            gold_1
        )
    )


    if response.startswith(
        "ERROR:"
    ):

        return pd.Series({

            "task_type":
                task_type,

            "task_aware_correct":
                0,

            "match_reason":
                "generation_error",
        })


    # --------------------------------------------------------
    # Numeric
    # --------------------------------------------------------

    if task_type == "numeric":

        correct = (
            numeric_answer_present(
                response,
                gold_1
            )
        )

        reason = (
            "primary_numeric_match"
            if correct
            else
            "numeric_mismatch"
        )


    # --------------------------------------------------------
    # Entity
    # --------------------------------------------------------

    elif task_type == "entity":

        if full_entity_present(
            response,
            gold_1
        ):

            correct = True

            reason = (
                "full_entity_match"
            )

        elif surname_with_support(
            response,
            gold_1,
            gold_2,
        ):

            correct = True

            reason = (
                "surname_plus_support"
            )

        else:

            correct = False

            reason = (
                "entity_mismatch"
            )


    # --------------------------------------------------------
    # Comparison
    # --------------------------------------------------------

    elif task_type == "compare":

        first = (
            numeric_answer_present(
                response,
                gold_1
            )
        )

        second = (
            numeric_answer_present(
                response,
                gold_2
            )
        )

        correct = (
            first and second
        )

        reason = (
            "both_comparison_values"
            if correct
            else
            "comparison_incomplete"
        )


    # --------------------------------------------------------
    # List
    # --------------------------------------------------------

    else:

        items = [
            clean_list_entity(
                item
            )
            for item
            in str(
                gold_1
            ).split(";")
            if str(
                item
            ).strip()
        ]


        present = [
            full_entity_present(
                response,
                item
            )
            for item
            in items
        ]


        correct = (
            len(items) > 0
            and
            all(present)
        )


        reason = (
            f"list_"
            f"{sum(present)}"
            f"of"
            f"{len(items)}"
        )


    return pd.Series({

        "task_type":
            task_type,

        "task_aware_correct":
            int(correct),

        "match_reason":
            reason,
    })


# ------------------------------------------------------------
# Regression tests for the original numeric bug
# ------------------------------------------------------------

assert (
    numeric_answer_present(
        "13.9",
        "11.8 rebounds per game"
    )
    is False
)

assert (
    numeric_answer_present(
        "11.8",
        "11.8 rebounds per game"
    )
    is True
)

assert (
    numeric_answer_present(
        "3",
        "2 retirements"
    )
    is False
)

print(
    "✓ Numeric regression tests passed"
)


# ------------------------------------------------------------
# Merge gold
# ------------------------------------------------------------

gold_columns = [
    c
    for c
    in [
        "query_id",
        "gold_answer_1",
        "gold_answer_2",
        "gold_answer_3",
    ]
    if c in track_U.columns
]


step1_scored = (
    raw_U.merge(

        track_U[
            gold_columns
        ],

        on="query_id",

        how="left",

        validate="many_to_one",
    )
)


score_columns = (
    step1_scored.apply(
        score_execution_row,
        axis=1,
    )
)


step1_scored = pd.concat(
    [
        step1_scored,
        score_columns,
    ],
    axis=1,
)


step1_scored.to_csv(
    OUT_DIR /
    "step1_execution_scored_task_aware_v7.csv",
    index=False,
)


# ------------------------------------------------------------
# Model summaries
# ------------------------------------------------------------

summary_rows = []


for model, model_df in (
    step1_scored.groupby(
        "model"
    )
):

    mean, low, high = (
        bootstrap_mean_ci(
            model_df[
                "task_aware_correct"
            ]
        )
    )


    summary_rows.append({

        "model":
            model,

        "n":
            len(model_df),

        "correct_n":
            int(
                model_df[
                    "task_aware_correct"
                ].sum()
            ),

        "accuracy":
            mean,

        "ci_low":
            low,

        "ci_high":
            high,
    })


step1_summary = (
    pd.DataFrame(
        summary_rows
    )
    .sort_values(
        "accuracy",
        ascending=False
    )
)


step1_summary.to_csv(
    OUT_DIR /
    "step1_execution_results_with_ci_v7.csv",
    index=False,
)


display(
    step1_summary.round(
        4
    )
)


# ------------------------------------------------------------
# Pairwise McNemar tests
# ------------------------------------------------------------

pairwise_rows = []


for model_a, model_b in combinations(
    REAL_MODELS,
    2,
):

    a = (
        step1_scored[
            step1_scored.model
            == model_a
        ][[
            "query_id",
            "task_aware_correct",
        ]]
        .rename(
            columns={
                "task_aware_correct":
                    "a"
            }
        )
    )


    b = (
        step1_scored[
            step1_scored.model
            == model_b
        ][[
            "query_id",
            "task_aware_correct",
        ]]
        .rename(
            columns={
                "task_aware_correct":
                    "b"
            }
        )
    )


    paired = a.merge(
        b,
        on="query_id",
        validate="one_to_one",
    )


    n10 = int(
        (
            (paired.a == 1)
            &
            (paired.b == 0)
        ).sum()
    )


    n01 = int(
        (
            (paired.a == 0)
            &
            (paired.b == 1)
        ).sum()
    )


    discordant = (
        n10 + n01
    )


    if discordant:

        p_value = binomtest(

            min(
                n10,
                n01
            ),

            n=discordant,

            p=0.5,

            alternative=
                "two-sided",
        ).pvalue

    else:

        p_value = 1.0


    difference, low, high = (
        bootstrap_diff(
            paired.a,
            paired.b
        )
    )


    pairwise_rows.append({

        "model_A":
            model_a,

        "model_B":
            model_b,

        "A_minus_B":
            difference,

        "ci_low":
            low,

        "ci_high":
            high,

        "mcnemar_n10":
            n10,

        "mcnemar_n01":
            n01,

        "mcnemar_exact_p":
            p_value,

        "matched_odds_ratio":
            (
                n10 + 0.5
            )
            /
            (
                n01 + 0.5
            ),
    })


step1_pairwise = pd.DataFrame(
    pairwise_rows
)


step1_pairwise[
    "p_holm"
] = holm_adjust(
    step1_pairwise[
        "mcnemar_exact_p"
    ].values
)


step1_pairwise[
    "significant_0.05"
] = (
    step1_pairwise[
        "p_holm"
    ]
    < 0.05
)


step1_pairwise.to_csv(
    OUT_DIR /
    "step1_execution_pairwise_inference_v7.csv",
    index=False,
)


display(
    step1_pairwise.round(
        4
    )
)


# Save higher-risk scoring cases for manual verification.
step1_scored[
    step1_scored.task_type.isin(
        [
            "list",
            "compare",
        ]
    )
    |
    step1_scored.match_reason.eq(
        "surname_plus_support"
    )
].to_csv(
    OUT_DIR /
    "step1_scoring_manual_audit_priority_v7.csv",
    index=False,
)


print("✓ STEP 1 COMPLETE")

In [ ]:
# ============================================================
# CELL 7 — TASK:
# STEP 2.
# Use cooperative_human_gold_final_480.csv as the PRIMARY
# cooperative evaluation. Do NOT rerun model responses here.
# ============================================================

score_fields = [
    "final_interpretation_validity",
    "final_assumption_quality",
    "final_answer_usefulness",
]


for column in score_fields:

    assert (
        human_gold[
            column
        ]
        .notna()
        .all()
    )

    assert (
        human_gold[
            column
        ]
        .between(
            0,
            2
        )
        .all()
    )


assert (
    human_gold[
        "human_score"
    ]
    .notna()
    .all()
)


step2_rows = []


for model, model_df in (
    human_gold.groupby(
        "model"
    )
):

    mean, low, high = (
        bootstrap_mean_ci(
            model_df[
                "human_score"
            ]
        )
    )


    step2_rows.append({

        "model":
            model,

        "n":
            len(model_df),

        "human_cooperative_score":
            mean,

        "ci_low":
            low,

        "ci_high":
            high,

        "interpretation_validity_norm":
            model_df[
                "final_interpretation_validity"
            ].mean() / 2,

        "assumption_quality_norm":
            model_df[
                "final_assumption_quality"
            ].mean() / 2,

        "answer_usefulness_norm":
            model_df[
                "final_answer_usefulness"
            ].mean() / 2,
    })


step2_summary = (
    pd.DataFrame(
        step2_rows
    )
    .sort_values(
        "human_cooperative_score",
        ascending=False,
    )
)


step2_summary.to_csv(
    OUT_DIR /
    "step2_cooperative_human_results_with_ci_v7.csv",
    index=False,
)


display(
    step2_summary.round(
        4
    )
)


# ------------------------------------------------------------
# Pairwise human cooperative inference
# ------------------------------------------------------------

pairwise_rows = []


for model_a, model_b in combinations(
    REAL_MODELS,
    2,
):

    a = (
        human_gold[
            human_gold.model
            == model_a
        ][[
            "query_id",
            "human_score",
        ]]
        .rename(
            columns={
                "human_score":
                    "a"
            }
        )
    )


    b = (
        human_gold[
            human_gold.model
            == model_b
        ][[
            "query_id",
            "human_score",
        ]]
        .rename(
            columns={
                "human_score":
                    "b"
            }
        )
    )


    paired = a.merge(
        b,
        on="query_id",
        validate="one_to_one",
    )


    difference, low, high = (
        bootstrap_diff(
            paired.a,
            paired.b,
        )
    )


    p_value = (
        paired_permutation_p(
            paired.a,
            paired.b,
        )
    )


    pairwise_rows.append({

        "model_A":
            model_a,

        "model_B":
            model_b,

        "A_minus_B":
            difference,

        "ci_low":
            low,

        "ci_high":
            high,

        "permutation_p":
            p_value,

        "paired_dz":
            paired_cohen_dz(
                paired.a,
                paired.b,
            ),
    })


step2_pairwise = pd.DataFrame(
    pairwise_rows
)


step2_pairwise[
    "p_holm"
] = holm_adjust(
    step2_pairwise[
        "permutation_p"
    ].values
)


step2_pairwise[
    "significant_0.05"
] = (
    step2_pairwise[
        "p_holm"
    ]
    < 0.05
)


step2_pairwise.to_csv(
    OUT_DIR /
    "step2_cooperative_human_pairwise_inference_v7.csv",
    index=False,
)


display(
    step2_pairwise.round(
        4
    )
)


# ------------------------------------------------------------
# Subdomain analysis
# ------------------------------------------------------------

step2_subdomain = (

    human_gold.groupby(
        [
            "model",
            "subdomain",
        ]
    )[
        "human_score"
    ]

    .agg(
        [
            "mean",
            "count",
        ]
    )

    .reset_index()
)


step2_subdomain.to_csv(
    OUT_DIR /
    "step2_human_by_subdomain_v7.csv",
    index=False,
)


# ------------------------------------------------------------
# Ambiguity-source analysis
# ------------------------------------------------------------

if (
    "final_primary_ambiguity_source"
    in human_gold.columns
):

    step2_ambiguity = (

        human_gold.groupby(
            [
                "model",
                "final_primary_ambiguity_source",
            ]
        )[
            "human_score"
        ]

        .agg(
            [
                "mean",
                "count",
            ]
        )

        .reset_index()
    )


    step2_ambiguity.to_csv(
        OUT_DIR /
        "step2_human_by_ambiguity_source_v7.csv",
        index=False,
    )


human_gold.to_csv(
    OUT_DIR /
    "step2_cooperative_human_gold_final_480_v7.csv",
    index=False,
)


print("✓ STEP 2 PRIMARY HUMAN EVALUATION COMPLETE")

In [ ]:
# ============================================================
# CELL 8 — TASK:
# STEP 2 supporting analysis.
# Recompute pre-adjudication human agreement if both original
# filled blinded annotation files are attached.
# ============================================================

if (
    ANNOTATOR_A_PATH
    and
    ANNOTATOR_B_PATH
):

    ann_a = pd.read_csv(
        ANNOTATOR_A_PATH
    )

    ann_b = pd.read_csv(
        ANNOTATOR_B_PATH
    )


    required_annotation_fields = [
        "blind_response_id",
        "interpretation_validity",
        "assumption_quality",
        "answer_usefulness",
    ]


    for column in (
        required_annotation_fields
    ):

        assert (
            column
            in ann_a.columns
        )

        assert (
            column
            in ann_b.columns
        )


    merged_annotations = (
        ann_a.merge(

            ann_b,

            on="blind_response_id",

            suffixes=(
                "_A",
                "_B"
            ),

            validate="one_to_one",
        )
    )


    assert (
        len(
            merged_annotations
        )
        == 480
    )


    agreement_rows = []


    dimensions = [
        "interpretation_validity",
        "assumption_quality",
        "answer_usefulness",
    ]


    for dimension in dimensions:

        a = merged_annotations[
            f"{dimension}_A"
        ].astype(int)

        b = merged_annotations[
            f"{dimension}_B"
        ].astype(int)


        agreement_rows.append({

            "dimension":
                dimension,

            "n":
                len(a),

            "exact_agreement":
                float(
                    (
                        a == b
                    ).mean()
                ),

            "weighted_cohen_kappa":
                float(
                    cohen_kappa_score(
                        a,
                        b,
                        weights="quadratic",
                    )
                ),

            "disagreement_count":
                int(
                    (
                        a != b
                    ).sum()
                ),
        })


    agreement_table = pd.DataFrame(
        agreement_rows
    )


    agreement_table.to_csv(
        OUT_DIR /
        "step2_pre_adjudication_human_agreement_v7.csv",
        index=False,
    )


    display(
        agreement_table.round(
            4
        )
    )

else:

    print(
        "Annotator A/B files not attached."
    )

    print(
        "Skipping agreement recomputation."
    )

In [ ]:
# ============================================================
# CELL 9 — TASK:
# STEP 3.
# Run the SAME interaction-decision prompt on ALL 360 queries.
# The model never sees the benchmark track label.
#
# Possible actions:
# ANSWER
# INTERPRET
# CLARIFY
# ============================================================

ACTION_SYSTEM = textwrap.dedent("""
You are a sports question-answering assistant being evaluated
on interaction decisions.

Your first task is to decide whether the USER'S REQUEST ITSELF
is sufficiently specified.

Choose exactly ONE action:

ANSWER
Use when the request is sufficiently specified to have a
definite intended answer.

Choose ANSWER even if you personally do not remember the
factual answer. Lack of factual knowledge is NOT ambiguity.

INTERPRET
Use when the request is somewhat underspecified, but one
conventional and defensible interpretation can make the
request useful without asking a follow-up question.

State the interpretation and assumptions explicitly.

CLARIFY
Use when essential information is missing and materially
different plausible choices would produce materially
different answers, so choosing one would be arbitrary.

Ask one concise clarification question.

The first non-empty line MUST be exactly one of:

Action: ANSWER
Action: INTERPRET
Action: CLARIFY

Do not use another action label.
""").strip()


def action_prompt(
    query
):

    return textwrap.dedent(
        f"""
        Query: {query}

        Decide the appropriate interaction action.

        If ANSWER:

        Action: ANSWER
        Answer: <brief response or brief factual uncertainty>

        If INTERPRET:

        Action: INTERPRET
        Interpretation: <one defensible interpretation>
        Assumptions: <explicit assumptions>
        Answer: <answer within that interpretation>

        If CLARIFY:

        Action: CLARIFY
        Clarification: <one concise question requesting the
        essential missing information>
        """
    ).strip()


action_bench = (
    df[[
        "query_id",
        "query_text",
        "subdomain",
        "final_query_type",
    ]]
    .copy()
    .reset_index(
        drop=True
    )
)


assert len(
    action_bench
) == 360


ACTION_CKPT = (
    CKPT_DIR /
    "step3_unified_action360_v7.json"
)


if ACTION_CKPT.exists():

    with open(
        ACTION_CKPT,
        "r"
    ) as f:

        action_responses = (
            json.load(f)
        )

else:

    action_responses = {
        model: [
            ""
        ] * len(action_bench)
        for model
        in REAL_MODELS
    }


for model in REAL_MODELS:

    values = (
        action_responses.get(
            model,
            []
        )
    )

    if len(values) < len(
        action_bench
    ):

        values += [
            ""
        ] * (
            len(action_bench)
            -
            len(values)
        )

    action_responses[
        model
    ] = values[
        :len(action_bench)
    ]


for model in REAL_MODELS:

    pending = [
        i
        for i, value
        in enumerate(
            action_responses[
                model
            ]
        )
        if not str(
            value
        ).strip()
    ]


    if not pending:

        print(
            f"✓ STEP 3 {model}: "
            "already complete"
        )

        continue


    load_local_model(
        model
    )


    for count, i in enumerate(

        tqdm(
            pending,
            desc=(
                f"STEP 3 — {model}"
            ),
            unit="query",
        ),

        start=1,
    ):

        query = action_bench.loc[
            i,
            "query_text"
        ]


        action_responses[
            model
        ][i] = call_local_text(

            model,

            ACTION_SYSTEM,

            action_prompt(
                query
            ),

            max_output_tokens=140,
        )


        if (
            count % 10 == 0
            or
            count == len(pending)
        ):

            with open(
                ACTION_CKPT,
                "w"
            ) as f:

                json.dump(
                    action_responses,
                    f,
                )


    clear_local_model()


with open(
    ACTION_CKPT,
    "w"
) as f:

    json.dump(
        action_responses,
        f,
    )


action_rows = []


for model in REAL_MODELS:

    for i, row in (
        action_bench.iterrows()
    ):

        action_rows.append({

            "model":
                model,

            "query_id":
                row["query_id"],

            "query_text":
                row["query_text"],

            "subdomain":
                row["subdomain"],

            "raw_response":
                action_responses[
                    model
                ][i],
        })


# ------------------------------------------------------------
# Constant clarification baseline
# ------------------------------------------------------------

for _, row in (
    action_bench.iterrows()
):

    action_rows.append({

        "model":
            "Always-Clarify-Baseline",

        "query_id":
            row["query_id"],

        "query_text":
            row["query_text"],

        "subdomain":
            row["subdomain"],

        "raw_response":
            (
                "Action: CLARIFY\n"
                "Clarification: Could you specify "
                "the essential missing information?"
            ),
    })


action_raw = pd.DataFrame(
    action_rows
)


action_raw.to_csv(
    OUT_DIR /
    "step3_raw_outputs_unified_action360_v7.csv",
    index=False,
)


print(
    "✓ STEP 3 generation complete"
)

print(
    action_raw.groupby(
        "model"
    ).size()
)

In [ ]:
# ============================================================
# CELL 10 — TASK:
# STEP 3 scoring.
# Calculate:
# - action accuracy
# - balanced accuracy
# - macro-F1
# - over-clarification
# - unsafe answering
# - appropriate clarification
# - paired McNemar tests
# - confusion matrices
# ============================================================

VALID_ACTIONS = [
    "ANSWER",
    "INTERPRET",
    "CLARIFY",
]


EXPECTED_ACTION = {

    "unambiguous":
        "ANSWER",

    "cooperative_ambiguous":
        "INTERPRET",

    "uncooperative":
        "CLARIFY",
}


def parse_action(
    response
):

    response = str(
        response
    )


    if response.startswith(
        "ERROR:"
    ):
        return "INVALID"


    match = re.search(

        r"(?im)^\s*"
        r"Action\s*:\s*"
        r"(ANSWER|INTERPRET|CLARIFY)"
        r"\b",

        response,
    )


    if not match:
        return "INVALID"


    return (
        match.group(1)
        .upper()
    )


gold_actions = (
    action_bench[[
        "query_id",
        "final_query_type",
    ]]
    .copy()
)


gold_actions[
    "expected_action"
] = (
    gold_actions[
        "final_query_type"
    ]
    .map(
        EXPECTED_ACTION
    )
)


action_eval = (
    action_raw.merge(

        gold_actions[[
            "query_id",
            "expected_action",
        ]],

        on="query_id",

        validate="many_to_one",
    )
)


action_eval[
    "predicted_action"
] = (
    action_eval[
        "raw_response"
    ]
    .map(
        parse_action
    )
)


action_eval[
    "action_correct"
] = (
    action_eval[
        "predicted_action"
    ]
    ==
    action_eval[
        "expected_action"
    ]
).astype(int)


action_eval.to_csv(
    OUT_DIR /
    "step3_all360_unified_action_classification_v7.csv",
    index=False,
)


def recall_for_action(
    model_df,
    action
):

    subset = model_df[
        model_df.expected_action
        == action
    ]

    return float(
        (
            subset.predicted_action
            == action
        ).mean()
    )


metric_rows = []


for model, model_df in (
    action_eval.groupby(
        "model"
    )
):

    recall_answer = (
        recall_for_action(
            model_df,
            "ANSWER"
        )
    )

    recall_interpret = (
        recall_for_action(
            model_df,
            "INTERPRET"
        )
    )

    recall_clarify = (
        recall_for_action(
            model_df,
            "CLARIFY"
        )
    )


    answerable = (
        model_df.expected_action
        .isin([
            "ANSWER",
            "INTERPRET",
        ])
    )


    uncooperative = (
        model_df.expected_action
        == "CLARIFY"
    )


    metric_rows.append({

        "model":
            model,

        "n":
            len(model_df),

        "action_accuracy":
            model_df[
                "action_correct"
            ].mean(),

        "balanced_accuracy":
            np.mean([
                recall_answer,
                recall_interpret,
                recall_clarify,
            ]),

        "macro_f1":
            f1_score(

                model_df[
                    "expected_action"
                ],

                model_df[
                    "predicted_action"
                ],

                labels=
                    VALID_ACTIONS,

                average=
                    "macro",

                zero_division=0,
            ),

        "recall_answer_U":
            recall_answer,

        "recall_interpret_C":
            recall_interpret,

        "appropriate_clarification_rate_X":
            recall_clarify,

        "over_clarification_rate_UC":
            (
                model_df.loc[
                    answerable,
                    "predicted_action"
                ]
                == "CLARIFY"
            ).mean(),

        "unsafe_answer_or_interpret_rate_X":
            (
                model_df.loc[
                    uncooperative,
                    "predicted_action"
                ]
                .isin([
                    "ANSWER",
                    "INTERPRET",
                ])
            ).mean(),

        "invalid_action_format_rate":
            (
                model_df[
                    "predicted_action"
                ]
                == "INVALID"
            ).mean(),
    })


step3_metrics = (
    pd.DataFrame(
        metric_rows
    )
    .sort_values(
        "action_accuracy",
        ascending=False,
    )
)


step3_metrics.to_csv(
    OUT_DIR /
    "step3_primary_unified_action_metrics_v7.csv",
    index=False,
)


display(
    step3_metrics.round(
        4
    )
)


# Always-clarify must be penalized.
constant_baseline = (
    step3_metrics[
        step3_metrics.model
        ==
        "Always-Clarify-Baseline"
    ]
    .iloc[0]
)


assert (
    constant_baseline[
        "action_accuracy"
    ]
    < 0.5
)


print(
    "✓ Always-Clarify accuracy:",
    round(
        constant_baseline[
            "action_accuracy"
        ],
        4
    )
)


# ------------------------------------------------------------
# Pairwise tests
# ------------------------------------------------------------

pairwise_rows = []


for model_a, model_b in combinations(
    REAL_MODELS,
    2,
):

    a = (
        action_eval[
            action_eval.model
            == model_a
        ][[
            "query_id",
            "action_correct",
        ]]
        .rename(
            columns={
                "action_correct":
                    "a"
            }
        )
    )


    b = (
        action_eval[
            action_eval.model
            == model_b
        ][[
            "query_id",
            "action_correct",
        ]]
        .rename(
            columns={
                "action_correct":
                    "b"
            }
        )
    )


    paired = a.merge(
        b,
        on="query_id",
        validate="one_to_one",
    )


    n10 = int(
        (
            (paired.a == 1)
            &
            (paired.b == 0)
        ).sum()
    )


    n01 = int(
        (
            (paired.a == 0)
            &
            (paired.b == 1)
        ).sum()
    )


    discordant = (
        n10 + n01
    )


    if discordant:

        p_value = binomtest(

            min(
                n10,
                n01
            ),

            n=discordant,

            p=0.5,

            alternative=
                "two-sided",
        ).pvalue

    else:

        p_value = 1.0


    difference, low, high = (
        bootstrap_diff(
            paired.a,
            paired.b
        )
    )


    pairwise_rows.append({

        "model_A":
            model_a,

        "model_B":
            model_b,

        "A_minus_B":
            difference,

        "ci_low":
            low,

        "ci_high":
            high,

        "mcnemar_n10":
            n10,

        "mcnemar_n01":
            n01,

        "mcnemar_exact_p":
            p_value,

        "matched_odds_ratio":
            (
                n10 + 0.5
            )
            /
            (
                n01 + 0.5
            ),
    })


step3_pairwise = pd.DataFrame(
    pairwise_rows
)


step3_pairwise[
    "p_holm"
] = holm_adjust(
    step3_pairwise[
        "mcnemar_exact_p"
    ].values
)


step3_pairwise[
    "significant_0.05"
] = (
    step3_pairwise[
        "p_holm"
    ]
    < 0.05
)


step3_pairwise.to_csv(
    OUT_DIR /
    "step3_unified_action_pairwise_inference_v7.csv",
    index=False,
)


display(
    step3_pairwise.round(
        4
    )
)


# ------------------------------------------------------------
# Confusion matrix data
# ------------------------------------------------------------

PREDICTED_ACTIONS = [
    "ANSWER",
    "INTERPRET",
    "CLARIFY",
    "INVALID",
]


confusion_rows = []


for model, model_df in (
    action_eval.groupby(
        "model"
    )
):

    for expected in VALID_ACTIONS:

        subset = model_df[
            model_df.expected_action
            == expected
        ]


        for predicted in (
            PREDICTED_ACTIONS
        ):

            confusion_rows.append({

                "model":
                    model,

                "expected_action":
                    expected,

                "predicted_action":
                    predicted,

                "count":
                    int(
                        (
                            subset.predicted_action
                            == predicted
                        ).sum()
                    ),

                "row_proportion":
                    float(
                        (
                            subset.predicted_action
                            == predicted
                        ).mean()
                    ),
            })


confusion = pd.DataFrame(
    confusion_rows
)


confusion.to_csv(
    OUT_DIR /
    "step3_unified_action_confusion_v7.csv",
    index=False,
)


# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

model_order = (
    REAL_MODELS
    +
    [
        "Always-Clarify-Baseline"
    ]
)


fig, axes = plt.subplots(
    1,
    4,
    figsize=(
        17,
        4
    ),
    sharey=True,
)


for ax, model in zip(
    axes,
    model_order
):

    model_conf = confusion[
        confusion.model
        == model
    ]


    matrix = np.array([

        [

            model_conf[
                (
                    model_conf.expected_action
                    == actual
                )
                &
                (
                    model_conf.predicted_action
                    == predicted
                )
            ]
            .row_proportion
            .iloc[0]

            for predicted
            in PREDICTED_ACTIONS

        ]

        for actual
        in VALID_ACTIONS
    ])


    ax.imshow(
        matrix,
        vmin=0,
        vmax=1,
        cmap="Greens",
        aspect="auto",
    )


    for i in range(3):

        for j in range(4):

            value = matrix[
                i,
                j
            ]

            ax.text(
                j,
                i,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=8,
                color=(
                    "white"
                    if value > 0.55
                    else "black"
                ),
            )


    ax.set_title(
        model,
        fontsize=8,
    )

    ax.set_xticks(
        range(4)
    )

    ax.set_xticklabels(
        [
            "Answer",
            "Interpret",
            "Clarify",
            "Invalid",
        ],
        rotation=30,
        ha="right",
    )

    ax.set_yticks(
        range(3)
    )

    ax.set_yticklabels(
        [
            "Answer",
            "Interpret",
            "Clarify",
        ]
    )


axes[0].set_ylabel(
    "Expected action"
)


fig.suptitle(
    "Unified interaction-action selection",
    fontweight="bold",
)


plt.tight_layout()


plt.savefig(
    OUT_DIR /
    "step3_fig_unified_action_confusion_v7.png",
    dpi=300,
    bbox_inches="tight",
)


plt.show()


print("✓ STEP 3 PRIMARY ANALYSIS COMPLETE")

In [ ]:
# ============================================================
# CELL 11 — TASK:
# STEP 3 secondary analysis.
# Rerun the 120 uncooperative queries using the original-style
# clarification/abstention prompt, then test robustness scores
# under abstention weights:
# 0, .25, .50, .75, 1.0
# ============================================================

LEGACY_SYSTEM = textwrap.dedent("""
You are a sports statistics assistant working in an
open-domain benchmark.

If a query is too underspecified to answer responsibly,
ask one concise clarification question or abstain briefly.

Do not guess.
""").strip()


def legacy_x_prompt(
    query
):

    return textwrap.dedent(
        f"""
        Query: {query}

        Do not guess.

        If clarification would make this answerable, output:

        Clarification: <one concise question>

        Otherwise output:

        Abstain: <brief reason>
        """
    ).strip()


LEGACY_CKPT = (
    CKPT_DIR /
    "step3_legacy_uncooperative120_v7.json"
)


if LEGACY_CKPT.exists():

    with open(
        LEGACY_CKPT,
        "r"
    ) as f:

        legacy_responses = (
            json.load(f)
        )

else:

    legacy_responses = {
        model: [
            ""
        ] * len(track_X)
        for model
        in REAL_MODELS
    }


for model in REAL_MODELS:

    values = (
        legacy_responses.get(
            model,
            []
        )
    )

    if len(values) < len(
        track_X
    ):

        values += [
            ""
        ] * (
            len(track_X)
            -
            len(values)
        )

    legacy_responses[
        model
    ] = values[
        :len(track_X)
    ]


for model in REAL_MODELS:

    pending = [
        i
        for i, value
        in enumerate(
            legacy_responses[
                model
            ]
        )
        if not str(
            value
        ).strip()
    ]


    if not pending:

        print(
            f"✓ Legacy X {model}: "
            "checkpoint complete"
        )

        continue


    load_local_model(
        model
    )


    for count, i in enumerate(

        tqdm(
            pending,
            desc=(
                f"Legacy X — {model}"
            ),
            unit="query",
        ),

        start=1,
    ):

        query = track_X.loc[
            i,
            "query_text"
        ]


        legacy_responses[
            model
        ][i] = call_local_text(

            model,

            LEGACY_SYSTEM,

            legacy_x_prompt(
                query
            ),

            max_output_tokens=80,
        )


        if (
            count % 10 == 0
            or
            count == len(pending)
        ):

            with open(
                LEGACY_CKPT,
                "w"
            ) as f:

                json.dump(
                    legacy_responses,
                    f,
                )


    clear_local_model()


with open(
    LEGACY_CKPT,
    "w"
) as f:

    json.dump(
        legacy_responses,
        f,
    )


legacy_rows = []


for model in REAL_MODELS:

    for i, row in (
        track_X.iterrows()
    ):

        legacy_rows.append({

            "model":
                model,

            "query_id":
                row["query_id"],

            "raw_response":
                legacy_responses[
                    model
                ][i],
        })


for _, row in (
    track_X.iterrows()
):

    legacy_rows.append({

        "model":
            "Heuristic-Baseline",

        "query_id":
            row["query_id"],

        "raw_response":
            (
                "Clarification: "
                "Which season, competition, "
                "or scope do you mean?"
            ),
    })


legacy_raw = pd.DataFrame(
    legacy_rows
)


def classify_legacy_behavior(
    response
):

    response = str(
        response
    ).strip()


    # IMPORTANT:
    # Explicit Abstain MUST be tested first.
    if re.match(
        r"(?i)^\s*abstain\s*:",
        response
    ):
        return "abstention"


    if re.match(
        r"(?i)^\s*clarification\s*:",
        response
    ):
        return "clarification"


    if re.search(

        r"(?i)"
        r"\b(?:cannot|can't|unable to)"
        r"\s+(?:determine|answer|say)\b"
        r"|"
        r"\binsufficient information\b"
        r"|"
        r"\bnot enough information\b",

        response,
    ):

        return "abstention"


    if (
        "?" in response
        and
        re.search(

            r"(?i)\b"
            r"(which|what|who|when|where|"
            r"could you|please specify|"
            r"do you mean)"
            r"\b",

            response,
        )
    ):

        return "clarification"


    return "other"


legacy_raw[
    "behavior"
] = (
    legacy_raw[
        "raw_response"
    ]
    .map(
        classify_legacy_behavior
    )
)


legacy_raw.to_csv(
    OUT_DIR /
    "step3_legacy_uncooperative_behavior_v7.csv",
    index=False,
)


display(
    pd.crosstab(
        legacy_raw.model,
        legacy_raw.behavior,
    )
)


sensitivity_rows = []


for abstention_weight in [
    0.00,
    0.25,
    0.50,
    0.75,
    1.00,
]:

    for model, model_df in (
        legacy_raw.groupby(
            "model"
        )
    ):

        scores = np.where(

            model_df.behavior
            == "clarification",

            1.0,

            np.where(

                model_df.behavior
                == "abstention",

                abstention_weight,

                0.0,
            )
        )


        sensitivity_rows.append({

            "abstention_weight":
                abstention_weight,

            "model":
                model,

            "legacy_weighted_robustness":
                float(
                    scores.mean()
                ),
        })


robustness_sensitivity = pd.DataFrame(
    sensitivity_rows
)


robustness_sensitivity.to_csv(
    OUT_DIR /
    "step3_legacy_robustness_weight_sensitivity_v7.csv",
    index=False,
)


display(
    robustness_sensitivity.pivot(

        index="model",

        columns=
            "abstention_weight",

        values=
            "legacy_weighted_robustness",
    ).round(3)
)


print(
    "✓ STEP 3 LEGACY SENSITIVITY COMPLETE"
)

In [ ]:
# ============================================================
# CELL 12 — TASK:
# STEP 4.
# Audit whether authentic source-table evidence exists.
#
# This cell is self-contained and SAFE:
# - It NEVER fabricates tables.
# - If no authentic table-evidence file exists, it records the
#   defensible manuscript decision to recast the experiment as
#   pre-retrieval / interaction-decision evaluation.
# - It ALWAYS creates step4_grounding_decision_v7.csv.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import glob
import os


# ============================================================
# 0. Recover basic variables if needed
# ============================================================

if "OUT_DIR" not in globals():

    OUT_DIR = Path(
        "/kaggle/working/ambitab_full_revision_v7"
    )

OUT_DIR = Path(OUT_DIR)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Recover benchmark dataframe if needed
# ------------------------------------------------------------

if "df" not in globals():

    benchmark_candidates = list(
        Path("/kaggle/input").rglob(
            "AmbiTab_Sports_Full_Gold_Benchmark_v3.csv"
        )
    )

    if not benchmark_candidates:

        benchmark_candidates = list(
            Path("/kaggle/input").rglob(
                "*AmbiTab*Gold*Benchmark*v3*.csv"
            )
        )

    if not benchmark_candidates:

        raise FileNotFoundError(
            "Could not find "
            "AmbiTab_Sports_Full_Gold_Benchmark_v3.csv"
        )

    BENCH_PATH = str(
        benchmark_candidates[0]
    )

    df = pd.read_csv(
        BENCH_PATH
    )

    print(
        "✓ Reloaded benchmark:",
        BENCH_PATH
    )


assert len(df) == 360


# ============================================================
# 1. Search for OPTIONAL authentic table-evidence file
# ============================================================

TABLE_EVIDENCE_PATH = None


possible_table_files = list(
    Path("/kaggle/input").rglob(
        "AmbiTab_Sports_Table_Evidence.csv"
    )
)


if not possible_table_files:

    possible_table_files = list(
        Path("/kaggle/input").rglob(
            "*AmbiTab*Table*Evidence*.csv"
        )
    )


if possible_table_files:

    TABLE_EVIDENCE_PATH = str(
        possible_table_files[0]
    )

    print(
        "✓ Authentic table evidence file found:",
        TABLE_EVIDENCE_PATH
    )

else:

    print(
        "No external authentic table-evidence file found."
    )


# ============================================================
# 2. Default Step-4 decision
# ============================================================

step4_result = {

    "total_queries":
        int(len(df)),

    "table_evidence_file":
        TABLE_EVIDENCE_PATH or "",

    "external_table_evidence_found":
        bool(TABLE_EVIDENCE_PATH),

    "table_evidence_coverage":
        0.0,

    "grounded_condition_available":
        False,

    "decision":
        "RECAST_PRE_RETRIEVAL",

    "recommended_scope":
        (
            "pre-retrieval interaction decisions "
            "for table-oriented sports question answering"
        ),

    "reason":
        (
            "The evaluated models were not provided "
            "with authentic source tables."
        ),
}


# ============================================================
# 3. If external table evidence exists, validate it
# ============================================================

if TABLE_EVIDENCE_PATH is not None:

    evidence = pd.read_csv(
        TABLE_EVIDENCE_PATH
    )


    required_columns = {
        "query_id",
        "table_evidence",
    }


    missing_columns = (
        required_columns
        -
        set(evidence.columns)
    )


    if missing_columns:

        print(
            "\nWARNING:"
        )

        print(
            "Table evidence file exists but is missing:",
            sorted(missing_columns)
        )

        step4_result[
            "decision"
        ] = (
            "RECAST_PRE_RETRIEVAL_"
            "INVALID_TABLE_EVIDENCE_FILE"
        )

        step4_result[
            "reason"
        ] = (
            "A table-evidence file was found, "
            "but required columns are missing."
        )

    else:

        evidence = (

            evidence[[
                "query_id",
                "table_evidence",
            ]]

            .drop_duplicates(
                "query_id"
            )
        )


        merged_evidence = (

            df[[
                "query_id"
            ]]

            .merge(
                evidence,
                on="query_id",
                how="left",
            )
        )


        nonempty = (

            merged_evidence[
                "table_evidence"
            ]

            .fillna("")

            .astype(str)

            .str.strip()

            .ne("")
        )


        coverage = float(
            nonempty.mean()
        )


        step4_result[
            "table_evidence_coverage"
        ] = coverage


        if coverage >= 0.95:

            step4_result[
                "grounded_condition_available"
            ] = True

            step4_result[
                "decision"
            ] = (
                "AUTHENTIC_TABLE_EVIDENCE_AVAILABLE"
            )

            step4_result[
                "recommended_scope"
            ] = (
                "A grounded evaluation condition "
                "can be developed separately."
            )

            step4_result[
                "reason"
            ] = (
                "Authentic table evidence is available "
                "for at least 95% of benchmark queries."
            )

        else:

            step4_result[
                "decision"
            ] = (
                "RECAST_PRE_RETRIEVAL_"
                "INSUFFICIENT_TABLE_COVERAGE"
            )

            step4_result[
                "reason"
            ] = (
                "Authentic table evidence coverage "
                "is below the 95% threshold."
            )


# ============================================================
# 4. Audit benchmark itself for possible embedded evidence
# ============================================================

evidence_keywords = [

    "table",
    "context",
    "evidence",
    "source",
    "markdown",
    "data_rows",
    "table_text",
]


candidate_columns = [

    column

    for column
    in df.columns

    if any(
        keyword
        in str(column).lower()
        for keyword
        in evidence_keywords
    )
]


embedded_rows = []


for column in candidate_columns:

    values = (

        df[column]

        .fillna("")

        .astype(str)

        .str.strip()
    )


    coverage = float(
        values.ne("").mean()
    )


    embedded_rows.append({

        "column":
            column,

        "nonempty_coverage":
            coverage,
    })


embedded_audit = pd.DataFrame(
    embedded_rows,
    columns=[
        "column",
        "nonempty_coverage",
    ],
)


embedded_audit.to_csv(
    OUT_DIR /
    "step4_embedded_grounding_column_audit_v7.csv",
    index=False,
)


# ============================================================
# 5. Save the Step-4 decision
# ============================================================

step4_table = pd.DataFrame([
    step4_result
])


STEP4_PATH = (
    OUT_DIR /
    "step4_grounding_decision_v7.csv"
)


step4_table.to_csv(
    STEP4_PATH,
    index=False,
)


# ============================================================
# 6. Write manuscript-facing scope statement
# ============================================================

if not step4_result[
    "grounded_condition_available"
]:

    scope_note = """
STEP 4 — GROUNDING AND SCOPE DECISION

The current AmbiTab-Sports experiments do not provide source
tables to the evaluated models.

Accordingly, the revised manuscript should not characterize
the reported experimental results as measurements of
table-grounded reasoning.

Instead, AmbiTab-Sports should be framed as evaluating the
pre-retrieval interaction decision that precedes evidence
access: whether a sports-data query should be answered
directly, answered under an explicit interpretation, or
clarified before retrieval.

The execution condition is retrieval-free. Consequently,
factual execution performance may depend on parametric model
knowledge and may be affected by memorization or training-data
contamination. Execution accuracy should therefore not be
interpreted as evidence of grounded table reasoning.

No synthetic or retrospectively fabricated table evidence was
created for this revision.
""".strip()

else:

    scope_note = """
STEP 4 — GROUNDING AND SCOPE DECISION

Authentic source-table evidence was detected for at least 95%
of benchmark queries. A separate grounded evaluation condition
may therefore be developed using those authentic sources.

The existing retrieval-free results should still be reported
separately from any future table-grounded condition.
""".strip()


with open(
    OUT_DIR /
    "step4_revised_scope_note.txt",
    "w",
    encoding="utf-8",
) as f:

    f.write(
        scope_note
    )


# ============================================================
# 7. Final validation
# ============================================================

assert STEP4_PATH.exists(), (
    "Step-4 decision file was not created."
)


reloaded_step4 = pd.read_csv(
    STEP4_PATH
)


assert len(
    reloaded_step4
) == 1


print(
    "\n=========================================="
)

print(
    "✓ STEP 4 COMPLETE"
)

print(
    "==========================================\n"
)


display(
    step4_table
)


print(
    "\nEmbedded evidence-like columns:"
)

if len(
    embedded_audit
):

    display(
        embedded_audit
    )

else:

    print(
        "None detected."
    )


print(
    "\nCreated:"
)

print(
    STEP4_PATH
)


print(
    "\nDecision:"
)

print(
    step4_result[
        "decision"
    ]
)


print(
    "\n✓ No table evidence was fabricated."
)

In [ ]:
# ============================================================
# CELL 13A — TASK:
# Load the FINAL human-verified Step-1 execution summary when
# available so the master table uses human correctness as the
# PRIMARY execution metric. The automatic scorer remains a
# secondary/reproducibility analysis.
# ============================================================

from pathlib import Path
import glob
import pandas as pd


def find_optional_file(filename):
    candidates = sorted(
        glob.glob(
            f"/kaggle/input/**/{filename}",
            recursive=True,
        )
    )

    if candidates:
        return Path(candidates[0])

    direct = OUT_DIR / filename
    if direct.exists():
        return direct

    working = Path("/kaggle/working") / filename
    if working.exists():
        return working

    return None


STEP1_HUMAN_RESULTS_PATH = find_optional_file(
    "step1_execution_human_results_with_ci.csv"
)

STEP1_HUMAN_GOLD_PATH = find_optional_file(
    "step1_execution_human_gold_final.csv"
)


if STEP1_HUMAN_RESULTS_PATH is not None:

    human_step1_summary = pd.read_csv(
        STEP1_HUMAN_RESULTS_PATH
    )

    required = {
        "model",
        "n",
        "accuracy",
        "ci_low",
        "ci_high",
    }

    missing = required - set(
        human_step1_summary.columns
    )

    if missing:
        raise KeyError(
            "Human Step-1 result file missing columns: "
            + str(sorted(missing))
        )

    # Only the three evaluated LLMs belong in the primary
    # human execution table.
    human_step1_summary = (
        human_step1_summary[
            human_step1_summary["model"].isin(
                REAL_MODELS
            )
        ]
        .copy()
        .reset_index(drop=True)
    )

    assert len(human_step1_summary) == 3
    assert set(human_step1_summary["model"]) == set(
        REAL_MODELS
    )

    # Existing Cell 13 expects the variable step1_summary.
    # Replace the automatic summary in memory with the final
    # human-verified summary before building the master table.
    step1_summary = human_step1_summary.copy()

    print(
        "✓ PRIMARY Step-1 metric: human-verified execution"
    )
    print(
        "Loaded:",
        STEP1_HUMAN_RESULTS_PATH,
    )

    display(
        step1_summary.round(4)
    )

else:

    print(
        "WARNING: step1_execution_human_results_with_ci.csv "
        "was not found. Cell 13 will fall back to the "
        "automatic Step-1 summary. For the final paper/repo, "
        "attach the human Step-1 result file."
    )


In [ ]:
# ============================================================
# CELL 13 — TASK:
# Build the FINAL master results table combining:
#
# STEP 1 = execution accuracy
# STEP 2 = adjudicated human cooperative evaluation
# STEP 3 = unified action-selection evaluation
#
# This version is self-repairing:
# If step1_summary, step2_summary, or step3_metrics are missing
# from memory, it rebuilds/loads them automatically.
#
# IMPORTANT:
# These capabilities remain separate.
# DO NOT average them into a single leaderboard score.
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd


# ============================================================
# 0. Basic checks
# ============================================================

if "OUT_DIR" not in globals():
    OUT_DIR = Path(
        "/kaggle/working/ambitab_full_revision_v7"
    )

OUT_DIR = Path(OUT_DIR)

REAL_MODELS = [
    "Qwen2.5-7B-Instruct",
    "Gemma-4-E4B-it",
    "Mistral-7B-Instruct-v0.2",
]


print("Preparing final master results table...\n")


# ============================================================
# 1. Recover STEP 1 summary if missing
# ============================================================

if "step1_summary" not in globals():

    step1_file = (
        OUT_DIR /
        "step1_execution_results_with_ci_v7.csv"
    )

    if step1_file.exists():

        step1_summary = pd.read_csv(
            step1_file
        )

        print(
            "✓ Loaded STEP 1 summary from:",
            step1_file.name
        )

    elif "step1_scored" in globals():

        rows = []

        for model, model_df in (
            step1_scored.groupby("model")
        ):

            if (
                "bootstrap_mean_ci"
                in globals()
            ):

                mean, low, high = (
                    bootstrap_mean_ci(
                        model_df[
                            "task_aware_correct"
                        ]
                    )
                )

            else:

                values = (
                    model_df[
                        "task_aware_correct"
                    ]
                    .astype(float)
                    .values
                )

                mean = float(
                    values.mean()
                )

                rng = (
                    np.random.default_rng(
                        42
                    )
                )

                boots = []

                for _ in range(5000):

                    sample = (
                        rng.choice(
                            values,
                            size=len(values),
                            replace=True,
                        )
                    )

                    boots.append(
                        sample.mean()
                    )

                low, high = (
                    np.quantile(
                        boots,
                        [
                            .025,
                            .975
                        ]
                    )
                )


            rows.append({

                "model":
                    model,

                "n":
                    len(model_df),

                "correct_n":
                    int(
                        model_df[
                            "task_aware_correct"
                        ].sum()
                    ),

                "accuracy":
                    mean,

                "ci_low":
                    low,

                "ci_high":
                    high,
            })


        step1_summary = pd.DataFrame(
            rows
        )

        print(
            "✓ Rebuilt STEP 1 summary"
        )

    else:

        raise RuntimeError(
            "\nSTEP 1 results are unavailable.\n"
            "Run CELL 6 first."
        )

else:

    print(
        "✓ STEP 1 summary already in memory"
    )


# ============================================================
# 2. Recover STEP 2 HUMAN cooperative summary if missing
# ============================================================

if "step2_summary" not in globals():

    step2_file = (
        OUT_DIR /
        "step2_cooperative_human_results_with_ci_v7.csv"
    )


    # --------------------------------------------------------
    # First try previously saved result
    # --------------------------------------------------------

    if step2_file.exists():

        step2_summary = pd.read_csv(
            step2_file
        )

        print(
            "✓ Loaded STEP 2 human summary from:",
            step2_file.name
        )


    # --------------------------------------------------------
    # Otherwise rebuild from human_gold
    # --------------------------------------------------------

    else:

        if "human_gold" not in globals():

            # Try to find original uploaded human gold file

            possible_human_files = list(
                Path("/kaggle/input").rglob(
                    "cooperative_human_gold_final_480.csv"
                )
            )

            if not possible_human_files:

                possible_human_files = list(
                    Path("/kaggle/input").rglob(
                        "*cooperative*human*gold*480*.csv"
                    )
                )


            if not possible_human_files:

                raise RuntimeError(
                    "\nSTEP 2 human gold file was not found.\n"
                    "Upload:\n"
                    "cooperative_human_gold_final_480.csv"
                )


            HUMAN_PATH = str(
                possible_human_files[0]
            )


            human_gold = pd.read_csv(
                HUMAN_PATH
            )


            print(
                "✓ Reloaded human gold:",
                HUMAN_PATH
            )


        print(
            "Human gold shape:",
            human_gold.shape
        )


        assert len(human_gold) == 480, (
            "Expected 480 human-scored responses, "
            f"found {len(human_gold)}."
        )


        required_human_columns = [
            "model",
            "query_id",
            "final_interpretation_validity",
            "final_assumption_quality",
            "final_answer_usefulness",
        ]


        missing_columns = [
            c
            for c in required_human_columns
            if c not in human_gold.columns
        ]


        if missing_columns:

            raise KeyError(
                "Human gold is missing columns: "
                + str(missing_columns)
            )


        # ----------------------------------------------------
        # Reconstruct total / normalized score if necessary
        # ----------------------------------------------------

        if (
            "human_total"
            not in human_gold.columns
        ):

            human_gold[
                "human_total"
            ] = (

                human_gold[
                    "final_interpretation_validity"
                ].astype(float)

                +

                human_gold[
                    "final_assumption_quality"
                ].astype(float)

                +

                human_gold[
                    "final_answer_usefulness"
                ].astype(float)
            )


        if (
            "human_score"
            not in human_gold.columns
        ):

            human_gold[
                "human_score"
            ] = (
                human_gold[
                    "human_total"
                ]
                / 6.0
            )


        # Validate scores
        assert (
            human_gold[
                "human_score"
            ]
            .between(
                0,
                1
            )
            .all()
        )


        # ----------------------------------------------------
        # Local bootstrap helper if Cell 4 unavailable
        # ----------------------------------------------------

        if (
            "bootstrap_mean_ci"
            not in globals()
        ):

            def bootstrap_mean_ci(
                values,
                n_boot=5000,
                seed=42,
            ):

                x = np.asarray(
                    values,
                    dtype=float
                )

                rng = (
                    np.random.default_rng(
                        seed
                    )
                )

                indices = (
                    rng.integers(
                        0,
                        len(x),
                        size=(
                            n_boot,
                            len(x)
                        ),
                    )
                )

                means = (
                    x[indices]
                    .mean(axis=1)
                )

                return (
                    float(
                        x.mean()
                    ),

                    float(
                        np.quantile(
                            means,
                            .025
                        )
                    ),

                    float(
                        np.quantile(
                            means,
                            .975
                        )
                    ),
                )


        # ----------------------------------------------------
        # Recalculate model-level human results
        # ----------------------------------------------------

        step2_rows = []


        for model, model_df in (
            human_gold.groupby(
                "model"
            )
        ):

            mean, low, high = (
                bootstrap_mean_ci(
                    model_df[
                        "human_score"
                    ]
                )
            )


            step2_rows.append({

                "model":
                    model,

                "n":
                    len(model_df),

                "human_cooperative_score":
                    mean,

                "ci_low":
                    low,

                "ci_high":
                    high,

                "interpretation_validity_norm":
                    (
                        model_df[
                            "final_interpretation_validity"
                        ]
                        .astype(float)
                        .mean()
                        / 2.0
                    ),

                "assumption_quality_norm":
                    (
                        model_df[
                            "final_assumption_quality"
                        ]
                        .astype(float)
                        .mean()
                        / 2.0
                    ),

                "answer_usefulness_norm":
                    (
                        model_df[
                            "final_answer_usefulness"
                        ]
                        .astype(float)
                        .mean()
                        / 2.0
                    ),
            })


        step2_summary = (
            pd.DataFrame(
                step2_rows
            )
            .sort_values(
                "human_cooperative_score",
                ascending=False,
            )
            .reset_index(
                drop=True
            )
        )


        step2_summary.to_csv(
            step2_file,
            index=False,
        )


        print(
            "✓ Rebuilt STEP 2 summary "
            "from final human gold"
        )


else:

    print(
        "✓ STEP 2 summary already in memory"
    )


# ============================================================
# 3. Recover STEP 3 summary if missing
# ============================================================

if "step3_metrics" not in globals():

    step3_file = (
        OUT_DIR /
        "step3_primary_unified_action_metrics_v7.csv"
    )


    if step3_file.exists():

        step3_metrics = pd.read_csv(
            step3_file
        )

        print(
            "✓ Loaded STEP 3 summary from:",
            step3_file.name
        )


    elif "action_eval" in globals():

        from sklearn.metrics import (
            f1_score
        )


        VALID_ACTIONS = [
            "ANSWER",
            "INTERPRET",
            "CLARIFY",
        ]


        def recall_for_action_local(
            data,
            action,
        ):

            subset = data[
                data.expected_action
                == action
            ]

            if len(subset) == 0:
                return np.nan

            return float(
                (
                    subset.predicted_action
                    == action
                ).mean()
            )


        rows = []


        for model, model_df in (
            action_eval.groupby(
                "model"
            )
        ):

            recall_answer = (
                recall_for_action_local(
                    model_df,
                    "ANSWER",
                )
            )

            recall_interpret = (
                recall_for_action_local(
                    model_df,
                    "INTERPRET",
                )
            )

            recall_clarify = (
                recall_for_action_local(
                    model_df,
                    "CLARIFY",
                )
            )


            answerable = (
                model_df.expected_action
                .isin([
                    "ANSWER",
                    "INTERPRET",
                ])
            )


            uncooperative = (
                model_df.expected_action
                == "CLARIFY"
            )


            rows.append({

                "model":
                    model,

                "n":
                    len(model_df),

                "action_accuracy":
                    model_df[
                        "action_correct"
                    ].mean(),

                "balanced_accuracy":
                    np.mean([
                        recall_answer,
                        recall_interpret,
                        recall_clarify,
                    ]),

                "macro_f1":
                    f1_score(

                        model_df[
                            "expected_action"
                        ],

                        model_df[
                            "predicted_action"
                        ],

                        labels=
                            VALID_ACTIONS,

                        average="macro",

                        zero_division=0,
                    ),

                "appropriate_clarification_rate_X":
                    recall_clarify,

                "over_clarification_rate_UC":
                    (
                        model_df.loc[
                            answerable,
                            "predicted_action"
                        ]
                        ==
                        "CLARIFY"
                    ).mean(),

                "unsafe_answer_or_interpret_rate_X":
                    (
                        model_df.loc[
                            uncooperative,
                            "predicted_action"
                        ]
                        .isin([
                            "ANSWER",
                            "INTERPRET",
                        ])
                    ).mean(),
            })


        step3_metrics = pd.DataFrame(
            rows
        )


        print(
            "✓ Rebuilt STEP 3 summary"
        )


    else:

        raise RuntimeError(
            "\nSTEP 3 results are unavailable.\n"
            "Run CELL 10 first."
        )


else:

    print(
        "✓ STEP 3 summary already in memory"
    )


# ============================================================
# 4. Display recovered source tables before merging
# ============================================================

print(
    "\nSTEP 1 summary:"
)

display(
    step1_summary.round(4)
)


print(
    "\nSTEP 2 HUMAN summary:"
)

display(
    step2_summary.round(4)
)


print(
    "\nSTEP 3 summary:"
)

display(
    step3_metrics.round(4)
)


# ============================================================
# 5. Check that all three real models exist
# ============================================================

for model in REAL_MODELS:

    assert model in set(
        step1_summary.model
    ), (
        f"{model} missing from STEP 1."
    )

    assert model in set(
        step2_summary.model
    ), (
        f"{model} missing from STEP 2."
    )

    assert model in set(
        step3_metrics.model
    ), (
        f"{model} missing from STEP 3."
    )


print(
    "\n✓ All three LLMs found "
    "in Steps 1, 2 and 3."
)


# ============================================================
# 6. Build final master results table
# ============================================================

master_rows = []


for model in REAL_MODELS:

    execution_row = (
        step1_summary[
            step1_summary.model
            == model
        ]
        .iloc[0]
    )


    cooperative_row = (
        step2_summary[
            step2_summary.model
            == model
        ]
        .iloc[0]
    )


    action_row = (
        step3_metrics[
            step3_metrics.model
            == model
        ]
        .iloc[0]
    )


    master_rows.append({

        "model":
            model,


        # ================================================
        # STEP 1 — EXECUTION
        # ================================================

        "execution_accuracy":
            execution_row[
                "accuracy"
            ],

        "execution_ci_low":
            execution_row[
                "ci_low"
            ],

        "execution_ci_high":
            execution_row[
                "ci_high"
            ],


        # ================================================
        # STEP 2 — HUMAN COOPERATIVE EVALUATION
        # ================================================

        "cooperative_human_score":
            cooperative_row[
                "human_cooperative_score"
            ],

        "cooperative_ci_low":
            cooperative_row[
                "ci_low"
            ],

        "cooperative_ci_high":
            cooperative_row[
                "ci_high"
            ],

        "interpretation_validity":
            cooperative_row[
                "interpretation_validity_norm"
            ],

        "assumption_quality":
            cooperative_row[
                "assumption_quality_norm"
            ],

        "answer_usefulness":
            cooperative_row[
                "answer_usefulness_norm"
            ],


        # ================================================
        # STEP 3 — INTERACTION DECISION
        # ================================================

        "action_accuracy":
            action_row[
                "action_accuracy"
            ],

        "balanced_action_accuracy":
            action_row[
                "balanced_accuracy"
            ],

        "action_macro_f1":
            action_row[
                "macro_f1"
            ],

        "appropriate_clarification_X":
            action_row[
                "appropriate_clarification_rate_X"
            ],

        "over_clarification_UC":
            action_row[
                "over_clarification_rate_UC"
            ],

        "unsafe_answer_or_interpret_X":
            action_row[
                "unsafe_answer_or_interpret_rate_X"
            ],
    })


master_results = (
    pd.DataFrame(
        master_rows
    )
)


# ============================================================
# 7. Save
# ============================================================

MASTER_PATH = (
    OUT_DIR /
    "FINAL_master_results_steps1_3_v7.csv"
)


master_results.to_csv(
    MASTER_PATH,
    index=False,
)


print(
    "\n=========================================="
)

print(
    "✓ FINAL MASTER TABLE CREATED"
)

print(
    "==========================================\n"
)


display(
    master_results.round(
        4
    )
)


print(
    "\nSaved to:"
)

print(
    MASTER_PATH
)

In [ ]:
# ============================================================
# CELL 14 — TASK:
# Generate and save the six publication/repository figures:
#
# Fig1_Benchmark_Three_Track_Framework.png
# Fig2_Subdomain_Distribution.png
# Fig3_Execution_Performance.png
# Fig4_Cooperative_Human_Evaluation.png
# Fig5_Action_Selection_Confusion_Matrix.png
# Fig6_Model_Behavior_Profile.png
#
# Figures are saved under OUT_DIR / "figures" and therefore
# are automatically included in the final results ZIP.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Short display names only for figure readability.
DISPLAY_NAME = {
    "Qwen2.5-7B-Instruct": "Qwen2.5-7B",
    "Gemma-4-E4B-it": "Gemma-4-E4B",
    "Mistral-7B-Instruct-v0.2": "Mistral-7B",
    "Heuristic-Baseline": "Heuristic",
    "Always-Clarify-Baseline": "Always-Clarify",
}


def _save(fig, filename):
    path = FIG_DIR / filename
    fig.savefig(
        path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)
    print("✓ Saved", path)
    return path


# ============================================================
# FIGURE 1 — Three-track benchmark framework
# ============================================================

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 6)
ax.axis("off")

# Input node
input_box = FancyBboxPatch(
    (0.45, 2.25),
    2.2,
    1.3,
    boxstyle="round,pad=0.04,rounding_size=0.08",
    linewidth=1.5,
    fill=False,
)
ax.add_patch(input_box)
ax.text(
    1.55,
    2.9,
    "Sports-data query",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
)

tracks = [
    (
        "Unambiguous",
        "Expected action: ANSWER\nPrimary evaluation:\nhuman-verified execution",
        4.15,
    ),
    (
        "Cooperative ambiguous",
        "Expected action: INTERPRET\nPrimary evaluation:\nhuman cooperative rubric",
        2.35,
    ),
    (
        "Uncooperative",
        "Expected action: CLARIFY\nPrimary evaluation:\naction selection / safety",
        0.55,
    ),
]

for title, subtitle, y in tracks:
    box = FancyBboxPatch(
        (4.0, y),
        4.1,
        1.25,
        boxstyle="round,pad=0.04,rounding_size=0.08",
        linewidth=1.5,
        fill=False,
    )
    ax.add_patch(box)
    ax.text(
        6.05,
        y + 0.83,
        title,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold",
    )
    ax.text(
        6.05,
        y + 0.39,
        subtitle,
        ha="center",
        va="center",
        fontsize=8.5,
    )
    arrow = FancyArrowPatch(
        (2.68, 2.9),
        (3.95, y + 0.62),
        arrowstyle="->",
        mutation_scale=14,
        linewidth=1.2,
    )
    ax.add_patch(arrow)

scope_box = FancyBboxPatch(
    (9.0, 1.65),
    2.45,
    2.45,
    boxstyle="round,pad=0.04,rounding_size=0.08",
    linewidth=1.5,
    fill=False,
)
ax.add_patch(scope_box)
ax.text(
    10.225,
    3.6,
    "Evaluation scope",
    ha="center",
    va="center",
    fontsize=11,
    fontweight="bold",
)
ax.text(
    10.225,
    2.78,
    "Pre-retrieval interaction\ndecisions for table-oriented\nsports QA\n\nNo source tables are\nprovided to the models",
    ha="center",
    va="center",
    fontsize=9,
)

for _, _, y in tracks:
    arrow = FancyArrowPatch(
        (8.12, y + 0.62),
        (8.95, 2.9),
        arrowstyle="->",
        mutation_scale=12,
        linewidth=1.0,
    )
    ax.add_patch(arrow)

ax.set_title(
    "AmbiTab-Sports: Three-Track Evaluation Framework",
    fontsize=15,
    fontweight="bold",
    pad=12,
)

_save(
    fig,
    "Fig1_Benchmark_Three_Track_Framework.png",
)


# ============================================================
# FIGURE 2 — Subdomain distribution by track
# ============================================================

track_order = [
    "unambiguous",
    "cooperative_ambiguous",
    "uncooperative",
]

track_labels = {
    "unambiguous": "Unambiguous",
    "cooperative_ambiguous": "Cooperative ambiguous",
    "uncooperative": "Uncooperative",
}

subdomain_counts = (
    df.groupby(
        ["subdomain", "final_query_type"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(columns=track_order, fill_value=0)
    .sort_index()
)

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(subdomain_counts.index))
width = 0.24

for j, track in enumerate(track_order):
    ax.bar(
        x + (j - 1) * width,
        subdomain_counts[track].values,
        width=width,
        label=track_labels[track],
    )

ax.set_xticks(x)
ax.set_xticklabels(
    subdomain_counts.index,
    rotation=25,
    ha="right",
)
ax.set_ylabel("Number of queries")
ax.set_xlabel("Sports subdomain")
ax.set_title(
    "AmbiTab-Sports Benchmark Distribution by Subdomain and Track"
)
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()

_save(
    fig,
    "Fig2_Subdomain_Distribution.png",
)


# ============================================================
# FIGURE 3 — Human-verified execution performance
# ============================================================

execution_plot = master_results[
    [
        "model",
        "execution_accuracy",
        "execution_ci_low",
        "execution_ci_high",
    ]
].copy()

execution_plot = execution_plot.sort_values(
    "execution_accuracy",
    ascending=False,
).reset_index(drop=True)

labels = [
    DISPLAY_NAME.get(x, x)
    for x in execution_plot["model"]
]

values = execution_plot[
    "execution_accuracy"
].to_numpy(dtype=float)

lower = values - execution_plot[
    "execution_ci_low"
].to_numpy(dtype=float)

upper = execution_plot[
    "execution_ci_high"
].to_numpy(dtype=float) - values

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.bar(
    np.arange(len(values)),
    values,
    yerr=np.vstack([lower, upper]),
    capsize=6,
)
ax.set_xticks(np.arange(len(values)))
ax.set_xticklabels(labels)
ax.set_ylim(0, min(1.0, max(0.5, float(np.max(values + upper)) + 0.08)))
ax.set_ylabel("Human-verified execution accuracy")
ax.set_title("Execution Performance on 120 Unambiguous Queries")
ax.grid(axis="y", alpha=0.25)

for i, value in enumerate(values):
    ax.text(
        i,
        value + 0.015,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

fig.tight_layout()

_save(
    fig,
    "Fig3_Execution_Performance.png",
)


# ============================================================
# FIGURE 4 — Cooperative human evaluation
# ============================================================

coop_plot = step2_summary[
    step2_summary["model"].isin(REAL_MODELS)
].copy()

coop_plot = coop_plot.sort_values(
    "human_cooperative_score",
    ascending=False,
).reset_index(drop=True)

labels = [
    DISPLAY_NAME.get(x, x)
    for x in coop_plot["model"]
]

values = coop_plot[
    "human_cooperative_score"
].to_numpy(dtype=float)

lower = values - coop_plot[
    "ci_low"
].to_numpy(dtype=float)

upper = coop_plot[
    "ci_high"
].to_numpy(dtype=float) - values

fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.bar(
    np.arange(len(values)),
    values,
    yerr=np.vstack([lower, upper]),
    capsize=6,
)
ax.set_xticks(np.arange(len(values)))
ax.set_xticklabels(labels)
ax.set_ylim(0, 1)
ax.set_ylabel("Normalized human cooperative score")
ax.set_title("Human Evaluation of Cooperative Interpretation")
ax.grid(axis="y", alpha=0.25)

for i, value in enumerate(values):
    ax.text(
        i,
        value + 0.02,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

fig.tight_layout()

_save(
    fig,
    "Fig4_Cooperative_Human_Evaluation.png",
)


# ============================================================
# FIGURE 5 — Unified action-selection confusion matrices
# ============================================================

# Recover confusion table if the in-memory dataframe is absent.
if "confusion" not in globals():
    confusion_path = OUT_DIR / "step3_unified_action_confusion_v7.csv"
    if not confusion_path.exists():
        confusion_path = OUT_DIR / "step3_unified_action_confusion_matrix.csv"
    if not confusion_path.exists():
        raise FileNotFoundError(
            "Step-3 confusion-matrix CSV was not found. Run Cell 10 first."
        )
    confusion = pd.read_csv(confusion_path)

model_order = REAL_MODELS + [
    "Always-Clarify-Baseline"
]
actuals = [
    "ANSWER",
    "INTERPRET",
    "CLARIFY",
]
predictions = [
    "ANSWER",
    "INTERPRET",
    "CLARIFY",
    "INVALID",
]

fig, axes = plt.subplots(
    1,
    4,
    figsize=(17, 4.7),
    sharey=True,
)

for ax, model in zip(axes, model_order):
    d = confusion[
        confusion["model"].eq(model)
    ]

    matrix = np.zeros((3, 4), dtype=float)

    for i, actual in enumerate(actuals):
        for j, predicted in enumerate(predictions):
            cell = d[
                d["expected_action"].eq(actual)
                & d["predicted_action"].eq(predicted)
            ]
            if len(cell):
                matrix[i, j] = float(
                    cell.iloc[0]["row_proportion"]
                )

    image = ax.imshow(
        matrix,
        vmin=0,
        vmax=1,
        aspect="auto",
    )

    for i in range(3):
        for j in range(4):
            ax.text(
                j,
                i,
                f"{matrix[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=8,
            )

    ax.set_title(
        DISPLAY_NAME.get(model, model),
        fontsize=9,
    )
    ax.set_xticks(range(4))
    ax.set_xticklabels(
        ["Answer", "Interpret", "Clarify", "Invalid"],
        rotation=35,
        ha="right",
    )
    ax.set_yticks(range(3))
    ax.set_yticklabels(
        ["Answer", "Interpret", "Clarify"]
    )
    ax.set_xlabel("Predicted action")

axes[0].set_ylabel("Expected action")
fig.colorbar(
    image,
    ax=axes,
    fraction=0.02,
    pad=0.02,
    label="Row proportion",
)
fig.suptitle(
    "Unified Interaction-Action Selection",
    fontsize=14,
    fontweight="bold",
)
fig.subplots_adjust(
    left=0.06,
    right=0.93,
    bottom=0.22,
    top=0.80,
    wspace=0.28,
)

_save(
    fig,
    "Fig5_Action_Selection_Confusion_Matrix.png",
)


# ============================================================
# FIGURE 6 — Descriptive model behavior profile
# ============================================================

profile = master_results.copy()
profile = profile[
    profile["model"].isin(REAL_MODELS)
].reset_index(drop=True)

profile_metrics = [
    ("execution_accuracy", "Execution\naccuracy"),
    ("cooperative_human_score", "Cooperative\nhuman score"),
    ("action_accuracy", "Action\naccuracy"),
    ("appropriate_clarification_X", "Appropriate\nclarification"),
    ("over_clarification_UC", "Over-\nclarification*"),
    ("unsafe_answer_or_interpret_X", "Unsafe answer/\ninterpret*"),
]

x = np.arange(len(profile_metrics))
width = 0.24

fig, ax = plt.subplots(figsize=(12, 6))

for j, (_, row) in enumerate(profile.iterrows()):
    vals = [
        float(row[col])
        for col, _ in profile_metrics
    ]
    ax.bar(
        x + (j - 1) * width,
        vals,
        width=width,
        label=DISPLAY_NAME.get(
            row["model"],
            row["model"],
        ),
    )

ax.set_xticks(x)
ax.set_xticklabels(
    [label for _, label in profile_metrics]
)
ax.set_ylim(0, 1)
ax.set_ylabel("Proportion / normalized score")
ax.set_title("Descriptive Model Behavior Profiles Across Evaluation Dimensions")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.25)
ax.text(
    0.0,
    -0.17,
    "* Lower values are preferable for over-clarification and unsafe answer/interpret rates. "
    "The figure is descriptive and is not an aggregate leaderboard.",
    transform=ax.transAxes,
    fontsize=9,
)
fig.tight_layout()

_save(
    fig,
    "Fig6_Model_Behavior_Profile.png",
)


# ============================================================
# Verify all six repository figures exist
# ============================================================

EXPECTED_FIGURES = [
    "Fig1_Benchmark_Three_Track_Framework.png",
    "Fig2_Subdomain_Distribution.png",
    "Fig3_Execution_Performance.png",
    "Fig4_Cooperative_Human_Evaluation.png",
    "Fig5_Action_Selection_Confusion_Matrix.png",
    "Fig6_Model_Behavior_Profile.png",
]

missing_figures = [
    name
    for name in EXPECTED_FIGURES
    if not (FIG_DIR / name).exists()
]

if missing_figures:
    raise RuntimeError(
        "Figure generation incomplete: "
        + str(missing_figures)
    )

print("\n✓ ALL SIX REPOSITORY FIGURES CREATED")
print("Figure directory:", FIG_DIR)


In [ ]:
# ============================================================
# CELL 15 — TASK:
# Validate the complete Step 1–4 run and create the final ZIP
# that you will download from Kaggle and upload back to ChatGPT.
# ============================================================

print(
    "\n========== FINAL SANITY CHECK ==========\n"
)


# ------------------------------------------------------------
# Step 1
# ------------------------------------------------------------

for model in REAL_MODELS:

    count = len(
        step1_scored[
            step1_scored.model
            == model
        ]
    )

    assert count == 120


print(
    "✓ STEP 1:"
    " 120 execution queries/model"
)


# ------------------------------------------------------------
# Step 2
# ------------------------------------------------------------

assert len(
    human_gold
) == 480

assert (
    human_gold.query_id.nunique()
    == 120
)


for model in (
    REAL_MODELS
    +
    [
        "Heuristic-Baseline"
    ]
):

    count = len(
        human_gold[
            human_gold.model
            == model
        ]
    )

    assert count == 120


print(
    "✓ STEP 2:"
    " 480 adjudicated human responses"
)


# ------------------------------------------------------------
# Step 3
# ------------------------------------------------------------

for model in REAL_MODELS:

    count = len(
        action_eval[
            action_eval.model
            == model
        ]
    )

    assert count == 360


baseline_accuracy = float(

    step3_metrics[
        step3_metrics.model
        ==
        "Always-Clarify-Baseline"
    ]

    .iloc[0][
        "action_accuracy"
    ]
)


assert (
    baseline_accuracy < 0.5
)


print(
    "✓ STEP 3:"
    " 360 unified-action queries/model"
)

print(
    "✓ Always-Clarify accuracy:",
    round(
        baseline_accuracy,
        4
    )
)


# ------------------------------------------------------------
# Step 4
# ------------------------------------------------------------

assert (
    OUT_DIR /
    "step4_grounding_decision_v7.csv"
).exists()


print(
    "✓ STEP 4:"
    " grounding/scope audit complete"
)




# ------------------------------------------------------------
# Figures
# ------------------------------------------------------------

EXPECTED_FIGURES = [
    "Fig1_Benchmark_Three_Track_Framework.png",
    "Fig2_Subdomain_Distribution.png",
    "Fig3_Execution_Performance.png",
    "Fig4_Cooperative_Human_Evaluation.png",
    "Fig5_Action_Selection_Confusion_Matrix.png",
    "Fig6_Model_Behavior_Profile.png",
]

FIG_DIR = OUT_DIR / "figures"

for figure_name in EXPECTED_FIGURES:
    assert (FIG_DIR / figure_name).exists(), (
        f"Missing repository figure: {figure_name}. "
        "Run Cell 14 before this final packaging cell."
    )

print("✓ FIGURES: all six repository figures present")


# ------------------------------------------------------------
# Generation errors
# ------------------------------------------------------------

step1_errors = int(
    step1_scored[
        "raw_response"
    ]
    .astype(str)
    .str.startswith(
        "ERROR:"
    )
    .sum()
)


step3_errors = int(
    action_eval[
        "raw_response"
    ]
    .astype(str)
    .str.startswith(
        "ERROR:"
    )
    .sum()
)


legacy_errors = int(
    legacy_raw[
        "raw_response"
    ]
    .astype(str)
    .str.startswith(
        "ERROR:"
    )
    .sum()
)


print(
    "\nGeneration errors"
)

print(
    "STEP 1:",
    step1_errors
)

print(
    "STEP 3 unified:",
    step3_errors
)

print(
    "STEP 3 legacy:",
    legacy_errors
)


# ------------------------------------------------------------
# Reproducibility metadata
# ------------------------------------------------------------

metadata = {

    "benchmark_file":
        Path(
            BENCH_PATH
        ).name,

    "human_gold_file":
        Path(
            HUMAN_PATH
        ).name,

    "total_benchmark_queries":
        len(df),

    "unambiguous_queries":
        len(track_U),

    "cooperative_queries":
        len(track_C),

    "uncooperative_queries":
        len(track_X),

    "human_scored_responses":
        len(human_gold),

    "primary_seed":
        PRIMARY_SEED,

    "bootstrap_resamples":
        N_BOOT,

    "permutation_resamples":
        N_PERM,

    "step1_generation_errors":
        step1_errors,

    "step3_generation_errors":
        step3_errors,

    "legacy_generation_errors":
        legacy_errors,

    "step4_decision":
        step4_result[
            "decision"
        ],

    "step4_table_coverage":
        step4_result[
            "table_evidence_coverage"
        ],
}


with open(
    OUT_DIR /
    "run_metadata_v7.json",
    "w"
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
    )


# ------------------------------------------------------------
# Final ZIP
# ------------------------------------------------------------

ZIP_PATH = Path(
    "/kaggle/working/"
    "AmbiTab_Full_Revision_Steps1_4_v7_RESULTS.zip"
)


if ZIP_PATH.exists():
    ZIP_PATH.unlink()


with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:

    for path in sorted(
        OUT_DIR.rglob("*")
    ):

        if not path.is_file():
            continue

        # Do not include resumable checkpoints
        if (
            "checkpoints"
            in path.parts
        ):
            continue

        archive.write(
            path,
            arcname=str(
                path.relative_to(
                    OUT_DIR
                )
            ),
        )


print(
    "\n========================================="
)

print(
    "✓ STEPS 1–4 COMPLETE"
)

print(
    "========================================="
)


print(
    "\nDownload this file:\n"
)

print(
    ZIP_PATH
)
